# CS781 Phase 3 — Experiment 3
## Model 2 (Phase-1 Decoder-only Causal LM) + LORMA — Rhetorical Role Prediction

**Goal:** a substantially different Phase-3 PEFT experiment using the exact Phase-1 **Model 2** checkpoint and a from-scratch **LORMA / LoRMA+ multiplicative low-rank adaptation**.

This notebook preserves the same Phase-3 experimental contract used in the earlier runs:
- train only on `CL_train + IT_train`;
- select the best checkpoint on combined `CL_dev + IT_dev` **13-label Macro-F1**;
- run the exact supplied local Phase-3 evaluator before any final refit;
- optionally tune transition-aware decoding on dev only;
- recreate a fresh Phase-1 Model 2 + LORMA model and refit on train+dev for exactly `BEST_EPOCH`;
- create **heldout**, **refit**, and **heldout+refit ensemble** test variants;
- structurally validate every private-test prediction;
- write `prediction.json` at the root of each submission ZIP.

### Experiment identity
**Phase-1 Model 2 + LORMA (Q/V-only multiplicative low-rank adaptation) + BiGRU sentence-classification head.**

The Phase-1 decoder backbone remains **causal**, exactly matching pretraining. LORMA is applied only to the **Q and V branches of the fused QKV projection** in every one of the 13 blocks. K, attention output projection, FFN weights, embeddings, and LayerNorm weights remain frozen.

The downstream **BiGRU task head** supplies bidirectional task context over the frozen/adapter-produced token representations without changing the backbone's causal attention regime. This follows the corrected Model-2 adaptation pattern that worked reliably in Phase 2, while the objective here is still sentence-level rhetorical-role classification.

### LORMA form
For each enabled frozen projection response `z = W0 x`, the adapter computes

`z' = z + (alpha / r) * B(A(dropout(z)))`

which is the efficient row-vector form of the multiplicative update

`W = (I + (alpha/r) BA) W0`.

### Runtime / reproducibility
- target-aware tokenization is cached once;
- deterministic sortish length-grouped training reduces padding;
- fused AdamW is used on CUDA when available;
- elapsed time + ETA are printed during training and refit;
- logits, history, and a compact experiment manifest are saved for later analysis.

> Attach your Phase-1 Model 2 best checkpoint (`dec_full_best*.pt`) and the exact Phase-1 `tokenizer.json`, then run top-to-bottom.


### v2 changes (efficiency + model-selection granularity)
- **Mid-epoch evaluation (`EVAL_PER_EPOCH=2`).** Phase-3 experiments so far peaked within the first epoch (Exp1 best epoch = 1), so selecting only at epoch ends can miss the best point. Dev is scored every half epoch; the best checkpoint stores the exact training position (`best_frac`).
- **Refit by training position, not whole epochs.** The train+dev refit now stops after the same *fraction* of an epoch (`REFIT_BATCHES`) on the same LR schedule, so it matches the selected checkpoint.
- **Throughput:** micro-batch 4 → 8 with accumulation 4 → 2 (same effective batch 16, fewer kernel launches), 8 → 6 epochs, patience counted in evaluations (4).
- **Collapse warning** in every dev evaluation (prints if >60% of predictions are one class).


### v3 review/fixes applied
I kept the useful v2 changes: **half-epoch dev evaluation**, **same effective batch size with micro-batch 8**, **collapse warnings**, and **fractional-position refit**.

I also tightened the implementation without changing the experiment:
- the selected checkpoint now stores the **exact batch/update position** as well as epoch fraction;
- refit uses an explicit **total target micro-batch count** derived from the selected fraction, avoiding ambiguous “per-epoch” bookkeeping;
- the final checkpoint/manifest record the exact selected and replayed training positions;
- refit progress/ETA messages now report the true total target;
- a guard asserts that the selected checkpoint position is valid before refit.

**Core experiment remains unchanged:** Phase-1 Model 2 + Q/V-only multiplicative LORMA (`r=4`, `alpha=32`) + BiGRU RR head, exact Phase-3 evaluator, same labels/data split/submission contract.


In [1]:
# ============================================================
# CELL 1 — Imports, seed, paths, and Experiment-3 config
# ============================================================
import os, re, gc, json, math, time, random, zipfile, subprocess, sys, hashlib
from pathlib import Path
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report
from huggingface_hub import HfApi, hf_hub_download, login

SEED = 1234
def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
seed_everything()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision("high")

HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    print("Could not read Kaggle HF_TOKEN secret:", repr(e))
assert HF_TOKEN, "HF_TOKEN not found in Kaggle Secrets."
login(token=HF_TOKEN)
HF_REPO = "Exploration-Lab/CS-781-Capstone"

ROOT = Path("/kaggle/working/phase3_exp3_model2_lorma")
ROOT.mkdir(parents=True, exist_ok=True)
BEST_CKPT = ROOT / "best_lorma_rr.pt"
FINAL_CKPT = ROOT / "final_refit_lorma_rr.pt"
DEV_PRED = ROOT / "prediction_dev.json"
CL_DEV_TRUTH = ROOT / "cl_dev_truth.json"
IT_DEV_TRUTH = ROOT / "it_dev_truth.json"
LABELS_JSON = ROOT / "labels.json"
EVAL_SCRIPT = ROOT / "evaluate_phase3.py"
OFFICIAL_PRED = ROOT / "prediction.json"
SUBMISSION_ZIP = ROOT / "submission.zip"

# -------- Exact Phase-1 Model 2 architecture --------
VOCAB_SIZE = 24_576
D_MODEL = 640
N_LAYERS = 13
N_HEADS = 10
FFN_DIM = 2560
MAX_POSITIONS = 2048
DROPOUT = 0.0
EXPECTED_TOKENIZER_SHA256 = "9cfbe6f0e49067f63ba71fe1013021c5de9269a2b4bc6c9ae30a72e42102106d"

# -------- Phase-3 context / task --------
MAX_LEN = 384
CONTEXT_SENTS = 1
USE_POS = True
TARGET_BUDGET = 250
PREV_BUDGET = 55
NEXT_BUDGET = 55

# -------- LORMA: corrected Model-2 pattern --------
# W = (I + alpha/r * B A) W0, applied to Q and V branches of fused QKV only.
LORMA_R = 4
LORMA_ALPHA = 32
LORMA_DROPOUT = 0.10
LORMA_ADAPT_Q = True
LORMA_ADAPT_K = False
LORMA_ADAPT_V = True
LORMA_LR = 2e-4

# -------- RR task head --------
# The backbone remains causal; this task head can use both directions.
HEAD_TYPE = "bigru"
HEAD_HIDDEN = 256
HEAD_DROPOUT = 0.20
HEAD_LR = 6e-4

# -------- Optimization --------
NUM_EPOCHS = 6
EARLY_STOP_PATIENCE = 4        # counted in dev evaluations (EVAL_PER_EPOCH per epoch)
EVAL_PER_EPOCH = 2
MICRO_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 16
GRAD_ACCUM_STEPS = 2
WEIGHT_DECAY = 0.01
GRAD_CLIP = 1.0
WARMUP_RATIO = 0.06
USE_AMP = True
LABEL_SMOOTHING = 0.03
CLASS_WEIGHT_MIN = 0.45
CLASS_WEIGHT_MAX = 2.50

PRINT_EVERY = 250
NUM_WORKERS = 2
LENGTH_GROUPED_TRAINING = True
LENGTH_BUCKET_MULT = 64
NAN_CHECK_EVERY = 50

# Refit replays the validated-run LR schedule (NUM_EPOCHS schedule, stopped at the selected training position)
# instead of annealing to zero early.
REFIT_MATCH_ORIGINAL_SCHEDULE = True
ENSEMBLE_HELDOUT_AND_REFIT = True

print("Output directory:", ROOT)
print(f"Model2: {N_LAYERS} layers | d={D_MODEL} | heads={N_HEADS} | FFN={FFN_DIM}")
print(f"LORMA: Q={LORMA_ADAPT_Q} K={LORMA_ADAPT_K} V={LORMA_ADAPT_V} | r={LORMA_R} alpha={LORMA_ALPHA} dropout={LORMA_DROPOUT}")
print("Backbone attention: causal (Phase-1 behavior preserved)")
print("Task head:", HEAD_TYPE, "| hidden:", HEAD_HIDDEN)
print("Context sentences each side:", CONTEXT_SENTS, "| MAX_LEN:", MAX_LEN)


Device: cuda
GPU: Tesla T4
Output directory: /kaggle/working/phase3_exp3_model2_lorma
Model2: 13 layers | d=640 | heads=10 | FFN=2560
LORMA: Q=True K=False V=True | r=4 alpha=32 dropout=0.1
Backbone attention: causal (Phase-1 behavior preserved)
Task head: bigru | hidden: 256
Context sentences each side: 1 | MAX_LEN: 384


In [2]:
# ============================================================
# CELL 2 — Find Phase-1 Model 2 checkpoint and tokenizer safely
# ============================================================
def find_files(predicate):
    out=[]
    for base in ["/kaggle/input", "/kaggle/working"]:
        if not os.path.isdir(base): continue
        for root, _, files in os.walk(base):
            for fn in files:
                p=os.path.join(root,fn)
                if predicate(fn,p): out.append(p)
    return out

ckpt_matches = find_files(
    lambda fn,p: fn.lower().endswith(".pt") and "dec_full_best" in fn.lower()
)
if not ckpt_matches:
    ckpt_matches = find_files(
        lambda fn,p: fn.lower().endswith(".pt")
        and "decoder" in p.lower()
        and "best" in fn.lower()
    )

tok_matches = find_files(
    lambda fn,p: fn.lower().endswith(".json") and "tokenizer" in fn.lower()
)

print("Checkpoint candidates:")
for p in ckpt_matches: print(" ", p)
print("Tokenizer candidates:")
for p in tok_matches: print(" ", p)

assert ckpt_matches, "Attach the Phase-1 Model2 best checkpoint (dec_full_best*.pt)."
assert tok_matches, "Attach the exact Phase-1 tokenizer.json."

def sha256_file(p):
    h=hashlib.sha256()
    with open(p,"rb") as f:
        for b in iter(lambda:f.read(1<<20), b""):
            h.update(b)
    return h.hexdigest()

def score_path(p, words):
    low=p.lower()
    return sum(w in low for w in words)

PHASE1_CKPT = sorted(
    ckpt_matches,
    key=lambda p:(score_path(p,["model2","decoder","phase1","dec_full_best"]),p),
    reverse=True
)[0]

exact_tok=[p for p in tok_matches if sha256_file(p)==EXPECTED_TOKENIZER_SHA256]
TOKENIZER_FILE = exact_tok[0] if exact_tok else sorted(tok_matches)[0]

print("Using checkpoint:", PHASE1_CKPT)
print("Using tokenizer :", TOKENIZER_FILE)
print("Tokenizer SHA256:", sha256_file(TOKENIZER_FILE))

assert sha256_file(TOKENIZER_FILE)==EXPECTED_TOKENIZER_SHA256, (
    "Tokenizer hash mismatch — attach the exact Phase-1 tokenizer."
)


Checkpoint candidates:
  /kaggle/input/datasets/aaryant23/phase2-model2-ckpts/dec_full_best (8).pt
Tokenizer candidates:
  /kaggle/input/datasets/aaryant23/phase2-model2-ckpts/tokenizer (1).json
Using checkpoint: /kaggle/input/datasets/aaryant23/phase2-model2-ckpts/dec_full_best (8).pt
Using tokenizer : /kaggle/input/datasets/aaryant23/phase2-model2-ckpts/tokenizer (1).json
Tokenizer SHA256: 9cfbe6f0e49067f63ba71fe1013021c5de9269a2b4bc6c9ae30a72e42102106d


In [4]:
# ============================================================
# CELL 3 — Original custom byte-level LegalBPE
# ============================================================

BPE_SPECIALS = ["<pad>", "<unk>", "<s>", "</s>"]

class LegalBPE:
    def __init__(self, vocab_size=5000):
        self.vocab_size = int(vocab_size)
        self.merges = []
        self.itos = list(BPE_SPECIALS)
        self.stoi = {t:i for i,t in enumerate(self.itos)}
        self.symbol_bytes = {}
        for b in range(256):
            s = f"{b:02x}"
            self.symbol_bytes[s] = bytes([b])
            self.stoi[s] = len(self.itos)
            self.itos.append(s)
        self._reset_encode_cache()

    @property
    def pad_token_id(self): return self.stoi["<pad>"]
    @property
    def unk_token_id(self): return self.stoi["<unk>"]
    @property
    def bos_token_id(self): return self.stoi["<s>"]
    @property
    def eos_token_id(self): return self.stoi["</s>"]
    def __len__(self): return len(self.itos)

    @staticmethod
    def word_to_symbols(word):
        return [f"{b:02x}" for b in word.encode("utf-8")]

    def _reset_encode_cache(self):
        self._ranks = {(l,r): i for i,(l,r) in enumerate(self.merges)}
        self._word_cache = {}
        self._cache_nmerges = len(self.merges)

    def encode_word(self, word):
        if self._cache_nmerges != len(self.merges):
            self._reset_encode_cache()

        hit = self._word_cache.get(word)
        if hit is not None:
            return list(hit)

        symbols = self.word_to_symbols(word)
        ranks = self._ranks

        while len(symbols) > 1:
            best = None
            best_rank = None
            for j in range(len(symbols)-1):
                r = ranks.get((symbols[j], symbols[j+1]))
                if r is not None and (best_rank is None or r < best_rank):
                    best = (symbols[j], symbols[j+1])
                    best_rank = r
            if best is None:
                break

            left, right = best
            merged = []
            j = 0
            while j < len(symbols):
                if j+1 < len(symbols) and symbols[j] == left and symbols[j+1] == right:
                    merged.append(left+right)
                    j += 2
                else:
                    merged.append(symbols[j])
                    j += 1
            symbols = merged

        if len(self._word_cache) >= 250_000:
            self._word_cache.clear()
        self._word_cache[word] = tuple(symbols)
        return symbols

    def encode(self, text):
        out = []
        unk = self.unk_token_id
        for unit in re.findall(r"\S+|\s+", text):
            for s in self.encode_word(unit):
                out.append(self.stoi.get(s, unk))
        return out

    def decode(self, ids):
        buf = bytearray()
        for idx in ids:
            idx = int(idx)
            tok = self.itos[idx]
            if tok in BPE_SPECIALS:
                continue
            buf.extend(self.symbol_bytes[tok])
        return bytes(buf).decode("utf-8", errors="replace")

    @classmethod
    def load_json(cls, path):
        obj = json.load(open(path, encoding="utf-8"))
        tok = cls(obj["vocab_size"])
        tok.merges = [list(x) for x in obj["merges"]]
        tok.itos = list(obj["itos"])
        tok.stoi = {t:i for i,t in enumerate(tok.itos)}
        tok.vocab_size = len(tok.itos)

        tok.symbol_bytes = {}
        for b in range(256):
            s = f"{b:02x}"
            tok.symbol_bytes[s] = bytes([b])
        for l, r in tok.merges:
            tok.symbol_bytes[l+r] = tok.symbol_bytes[l] + tok.symbol_bytes[r]

        tok._reset_encode_cache()
        return tok

tokenizer = LegalBPE.load_json(TOKENIZER_FILE)
print("Tokenizer vocab:", len(tokenizer))
assert len(tokenizer) == VOCAB_SIZE, (len(tokenizer), VOCAB_SIZE)

probe = "IN THE SUPREME COURT OF INDIA\nSection 14: legal test."
assert tokenizer.decode(tokenizer.encode(probe)) == probe
print("✅ tokenizer round-trip passed")


Tokenizer vocab: 24576
✅ tokenizer round-trip passed


In [5]:
# ============================================================
# CELL 4 — Exact Phase-1 Model 2 decoder-only architecture
# ============================================================
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model, max_positions):
        super().__init__()
        pos = torch.arange(max_positions, dtype=torch.float32).unsqueeze(1)
        div = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32)
            * (-math.log(10000.0) / d_model)
        )
        pe = torch.zeros(max_positions, d_model, dtype=torch.float32)
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe, persistent=False)

    def forward(self, position_ids):
        return self.pe[position_ids]


class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.dropout = dropout

    def forward(self, x, past_kv=None, use_cache=False, key_valid=None):
        B, T, C = x.shape

        qkv = self.qkv(x).view(B, T, 3, self.n_heads, self.head_dim)
        q, k, v = qkv.unbind(dim=2)
        q = q.transpose(1, 2)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)

        past_len = 0
        if past_kv is not None:
            pk, pv = past_kv
            past_len = pk.size(2)
            k = torch.cat([pk, k], dim=2)
            v = torch.cat([pv, v], dim=2)

        attn_mask = None
        is_causal = False

        if key_valid is not None:
            key_mask = key_valid[:, None, None, :].bool()
            if past_len == 0 and T > 1:
                causal = torch.ones((T, T), dtype=torch.bool, device=x.device).tril()
                attn_mask = key_mask & causal[None, None, :, :]
                # Protect fully-masked padded query rows without changing real-token attention.
                eye = torch.eye(T, dtype=torch.bool, device=x.device)[None, None, :, :]
                attn_mask = attn_mask | eye
            else:
                attn_mask = key_mask
        elif past_kv is None:
            is_causal = True

        y = F.scaled_dot_product_attention(
            q, k, v,
            attn_mask=attn_mask,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=is_causal,
        )

        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.out_proj(y)

        new_kv = (k, v) if use_cache else None
        return y, new_kv


class DecoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, ffn_dim, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, ffn_dim),
            nn.GELU(),
            nn.Linear(ffn_dim, d_model),
            nn.Dropout(dropout),
        )
        self.resid_drop = nn.Dropout(dropout)

    def forward(self, x, past_kv=None, use_cache=False, key_valid=None):
        a, new_kv = self.attn(
            self.ln1(x),
            past_kv=past_kv,
            use_cache=use_cache,
            key_valid=key_valid,
        )
        x = x + self.resid_drop(a)
        x = x + self.ff(self.ln2(x))
        return x, new_kv


class DecoderOnlyLM(nn.Module):
    def __init__(
        self, vocab_size, d_model, n_layers, n_heads,
        ffn_dim, dropout, max_positions
    ):
        super().__init__()
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.max_positions = max_positions

        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos = SinusoidalPositionalEncoding(d_model, max_positions)
        self.drop = nn.Dropout(dropout)

        self.blocks = nn.ModuleList([
            DecoderBlock(d_model, n_heads, ffn_dim, dropout)
            for _ in range(n_layers)
        ])
        self.final_ln = nn.LayerNorm(d_model)

        self.apply(self._init_weights)

        scaled_std = 0.02 / math.sqrt(2 * n_layers)
        for block in self.blocks:
            nn.init.normal_(block.attn.out_proj.weight, mean=0.0, std=scaled_std)
            nn.init.normal_(block.ff[2].weight, mean=0.0, std=scaled_std)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def forward_hidden(
        self, input_ids, past_key_values=None, use_cache=False,
        position_ids=None, key_valid=None
    ):
        B, T = input_ids.shape
        past_len = 0 if past_key_values is None else past_key_values[0][0].size(2)

        if past_len + T > self.max_positions:
            raise ValueError(
                f"context {past_len + T} exceeds max_positions={self.max_positions}"
            )

        if position_ids is None:
            position_ids = torch.arange(
                past_len, past_len + T, device=input_ids.device
            )[None, :].expand(B, T)

        x = self.token_emb(input_ids) + self.pos(position_ids).to(self.token_emb.weight.dtype)
        x = self.drop(x)

        new_cache = [] if use_cache else None
        for i, block in enumerate(self.blocks):
            past = None if past_key_values is None else past_key_values[i]
            x, kv = block(
                x,
                past_kv=past,
                use_cache=use_cache,
                key_valid=key_valid,
            )
            if use_cache:
                new_cache.append(kv)

        x = self.final_ln(x)
        return x, new_cache

    def project(self, hidden):
        return F.linear(hidden, self.token_emb.weight)


def build_backbone():
    return DecoderOnlyLM(
        VOCAB_SIZE, D_MODEL, N_LAYERS, N_HEADS,
        FFN_DIM, DROPOUT, MAX_POSITIONS
    )

backbone = build_backbone()
n_backbone = sum(p.numel() for p in backbone.parameters())
print("Backbone params:", f"{n_backbone/1e6:.3f}M")
assert 75_000_000 <= n_backbone <= 85_000_000, n_backbone


Backbone params: 79.736M


In [6]:
# ============================================================
# CELL 5 — Load Phase-1 Model 2 checkpoint strictly
# ============================================================
obj = torch.load(PHASE1_CKPT, map_location="cpu", weights_only=False)
print("Checkpoint keys:", list(obj.keys()) if isinstance(obj, dict) else type(obj))
state = obj["model"] if isinstance(obj, dict) and "model" in obj else obj

missing, unexpected = backbone.load_state_dict(state, strict=True)
assert not missing and not unexpected
print("✅ strict=True checkpoint load succeeded")

if isinstance(obj, dict):
    print(
        "step:", obj.get("global_step", obj.get("step")),
        "val_loss:", obj.get("val_loss", obj.get("best_val"))
    )
    cfg = obj.get("config", {})
    if cfg:
        print("Checkpoint architecture:", cfg.get("architecture"))
        for k, expected in [
            ("d_model", D_MODEL),
            ("n_layers", N_LAYERS),
            ("n_heads", N_HEADS),
            ("ffn_dim", FFN_DIM),
            ("max_positions", MAX_POSITIONS),
        ]:
            if k in cfg:
                assert int(cfg[k]) == int(expected), (k, cfg[k], expected)

backbone = backbone.to(DEVICE)


Checkpoint keys: ['model', 'global_step', 'val_loss', 'config']
✅ strict=True checkpoint load succeeded
step: 16266 val_loss: 1.7356935616699047
Checkpoint architecture: decoder_only_causal


In [7]:
# ============================================================
# CELL 6 — LORMA from scratch: multiplicative low-rank Q/V adaptation
# ============================================================
class FusedQKVSelectiveLoRMAPlusLinear(nn.Module):
    """
    Selective multiplicative low-rank adaptation for fused QKV.

    Frozen base projection first produces [Q0,K0,V0].
    For each enabled branch j:
        y_j = z_j + (alpha/r) * B_j(A_j(dropout(z_j)))
    which is equivalent to:
        W_j = (I + (alpha/r) B_j A_j) W0_j

    K remains exactly frozen in this experiment.
    """
    def __init__(
        self, base, d_model, r=LORMA_R, alpha=LORMA_ALPHA,
        dropout=LORMA_DROPOUT,
        adapt_q=LORMA_ADAPT_Q,
        adapt_k=LORMA_ADAPT_K,
        adapt_v=LORMA_ADAPT_V,
    ):
        super().__init__()
        assert isinstance(base, nn.Linear)
        assert base.out_features == 3 * d_model
        assert r > 0

        self.base = base
        self.d_model = int(d_model)
        self.r = int(r)
        self.alpha = float(alpha)
        self.scale = self.alpha / self.r
        self.lorma_dropout = nn.Dropout(dropout)
        self.adapt = (bool(adapt_q), bool(adapt_k), bool(adapt_v))

        self.lorma_A = nn.ParameterDict()
        self.lorma_B = nn.ParameterDict()

        for enabled, name in zip(self.adapt, ("q", "k", "v")):
            if enabled:
                A = nn.Parameter(torch.empty(self.r, self.d_model))
                B = nn.Parameter(torch.zeros(self.d_model, self.r))
                nn.init.kaiming_uniform_(A, a=math.sqrt(5))
                self.lorma_A[name] = A
                self.lorma_B[name] = B

        for p in self.base.parameters():
            p.requires_grad = False

    def forward(self, x):
        # Bias-free base response first; frozen bias is added only once at the end.
        z = F.linear(x, self.base.weight, None)
        zs = list(z.split(self.d_model, dim=-1))
        outs = []

        for enabled, name, zj in zip(self.adapt, ("q", "k", "v"), zs):
            if enabled:
                A = self.lorma_A[name]
                B = self.lorma_B[name]
                delta = F.linear(
                    F.linear(self.lorma_dropout(zj), A),
                    B,
                )
                zj = zj + self.scale * delta
            outs.append(zj)

        y = torch.cat(outs, dim=-1)
        if self.base.bias is not None:
            y = y + self.base.bias
        return y


def inject_lorma(bb):
    # Freeze every original Phase-1 parameter first.
    for p in bb.parameters():
        p.requires_grad = False

    injected = 0
    for block in bb.blocks:
        block.attn.qkv = FusedQKVSelectiveLoRMAPlusLinear(
            block.attn.qkv,
            d_model=D_MODEL,
        )
        injected += 1

    lorma = [(n,p) for n,p in bb.named_parameters() if "lorma_" in n]
    expected_tensors = N_LAYERS * 4  # A_q, B_q, A_v, B_v
    assert len(lorma) == expected_tensors, (len(lorma), expected_tensors)
    assert all(p.requires_grad for _,p in lorma)

    expected_params = N_LAYERS * 2 * (LORMA_R * D_MODEL + D_MODEL * LORMA_R)
    actual_params = sum(p.numel() for _,p in lorma)
    assert actual_params == expected_params, (actual_params, expected_params)

    print("Injected fused-QKV LORMA modules:", injected)
    print("Trainable LORMA tensors:", len(lorma))
    print("LORMA trainable params:", actual_params)
    print("Expected LORMA params:", expected_params)

    # Zero-B initialization must make the injected backbone exactly equivalent
    # to the frozen Phase-1 backbone before any update.
    assert all(
        torch.count_nonzero(p.detach()).item() == 0
        for n,p in lorma if ".lorma_B." in n
    ), "LoRMA B must start at zero."

    return bb

backbone = inject_lorma(backbone)


Injected fused-QKV LORMA modules: 13
Trainable LORMA tensors: 52
LORMA trainable params: 133120
Expected LORMA params: 133120


In [8]:
# ============================================================
# CELL 7 — Download Phase-3 raw JSON files (bypass datasets CastError)
# ============================================================

api = HfApi()
repo_files = api.list_repo_files(
    repo_id=HF_REPO,
    repo_type="dataset",
    token=HF_TOKEN,
)

def exact_repo_file(filename):
    matches = [
        f for f in repo_files
        if os.path.basename(f).lower() == filename.lower()
    ]
    assert len(matches) == 1, (filename, matches)
    return matches[0]

FILES = {
    "CL_train": exact_repo_file("CL_train.json"),
    "CL_dev": exact_repo_file("CL_dev.json"),
    "IT_train": exact_repo_file("IT_train.json"),
    "IT_dev": exact_repo_file("IT_dev.json"),
    "CL_test": exact_repo_file("CL_test.json"),
    "IT_test": exact_repo_file("IT_test.json"),
    "label_vocab": exact_repo_file("label_vocab.json"),
}

print(json.dumps(FILES, indent=2))

def load_repo_json(repo_path):
    p = hf_hub_download(
        repo_id=HF_REPO,
        filename=repo_path,
        repo_type="dataset",
        token=HF_TOKEN,
    )
    with open(p, "r", encoding="utf-8") as f:
        return json.load(f)

cl_train_raw = load_repo_json(FILES["CL_train"])
cl_dev_raw   = load_repo_json(FILES["CL_dev"])
it_train_raw = load_repo_json(FILES["IT_train"])
it_dev_raw   = load_repo_json(FILES["IT_dev"])
cl_test_raw  = load_repo_json(FILES["CL_test"])
it_test_raw  = load_repo_json(FILES["IT_test"])
label_vocab_raw = load_repo_json(FILES["label_vocab"])

assert isinstance(cl_train_raw, list) and isinstance(it_train_raw, list)
assert isinstance(cl_dev_raw, list) and isinstance(it_dev_raw, list)
assert isinstance(cl_test_raw, dict) and isinstance(it_test_raw, dict)

print("CL train/dev:", len(cl_train_raw), len(cl_dev_raw))
print("IT train/dev:", len(it_train_raw), len(it_dev_raw))
print("CL/IT test docs:", len(cl_test_raw), len(it_test_raw))


{
  "CL_train": "Phase_3_RR/Train/CL_train.json",
  "CL_dev": "Phase_3_RR/Train/CL_dev.json",
  "IT_train": "Phase_3_RR/Train/IT_train.json",
  "IT_dev": "Phase_3_RR/Train/IT_dev.json",
  "CL_test": "Phase_3_RR/Test/CL_test.json",
  "IT_test": "Phase_3_RR/Test/IT_test.json",
  "label_vocab": "Phase_3_RR/Train/label_vocab.json"
}


CL_train.json:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

CL_dev.json:   0%|          | 0.00/336k [00:00<?, ?B/s]

IT_train.json:   0%|          | 0.00/1.51M [00:00<?, ?B/s]

IT_dev.json:   0%|          | 0.00/165k [00:00<?, ?B/s]

CL_test.json:   0%|          | 0.00/281k [00:00<?, ?B/s]

IT_test.json:   0%|          | 0.00/194k [00:00<?, ?B/s]

label_vocab.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

CL train/dev: 40 5
IT train/dev: 40 5
CL/IT test docs: 5 5


In [9]:
# ============================================================
# CELL 8 — Labels, examples, class weights, and data checks
# ============================================================

def parse_label_vocab(data):
    if isinstance(data, list):
        labels = list(data)
        label_to_id = {x:i for i,x in enumerate(labels)}
    elif isinstance(data, dict):
        vals = list(data.values())
        if all(isinstance(v, int) and not isinstance(v, bool) for v in vals):
            label_to_id = {str(k): int(v) for k, v in data.items()}
            assert sorted(label_to_id.values()) == list(range(len(label_to_id)))
            labels = [None] * len(label_to_id)
            for lab, idx in label_to_id.items():
                labels[idx] = lab
        elif all(isinstance(v, str) for v in vals):
            tmp = {int(k): v for k, v in data.items()}
            labels = [tmp[i] for i in sorted(tmp)]
            label_to_id = {x:i for i,x in enumerate(labels)}
        else:
            raise ValueError("Unsupported label_vocab format")
    else:
        raise ValueError("Unsupported label_vocab format")

    return labels, label_to_id

LABELS, LABEL_TO_ID = parse_label_vocab(label_vocab_raw)
ID_TO_LABEL = {v:k for k,v in LABEL_TO_ID.items()}
NUM_LABELS = len(LABELS)

assert NUM_LABELS == 13
print("Labels:")
for i, x in enumerate(LABELS):
    print(i, x)

def normalize_id(doc_id):
    doc_id = str(doc_id)
    return doc_id[:-4] if doc_id.endswith(".txt") else doc_id

def validate_labeled_docs(docs, name):
    ids = set()
    n_sent = 0
    for d in docs:
        did = normalize_id(d["id"])
        assert did not in ids, f"{name}: duplicate ID {did}"
        ids.add(did)
        assert isinstance(d["text"], list)
        assert isinstance(d["labels"], list)
        assert len(d["text"]) == len(d["labels"]), did
        assert all(x in LABEL_TO_ID for x in d["labels"]), did
        n_sent += len(d["text"])
    print(f"{name}: docs={len(docs)} sentences={n_sent}")
    return ids

cltr_ids = validate_labeled_docs(cl_train_raw, "CL_train")
cldv_ids = validate_labeled_docs(cl_dev_raw, "CL_dev")
ittr_ids = validate_labeled_docs(it_train_raw, "IT_train")
itdv_ids = validate_labeled_docs(it_dev_raw, "IT_dev")

assert not (cltr_ids & cldv_ids)
assert not (ittr_ids & itdv_ids)
# No document may silently appear in both legal domains/splits after `.txt` normalization.
assert not (cltr_ids & ittr_ids), "CL_train / IT_train ID overlap"
assert not (cltr_ids & itdv_ids), "CL_train / IT_dev ID overlap"
assert not (cldv_ids & ittr_ids), "CL_dev / IT_train ID overlap"
assert not (cldv_ids & itdv_ids), "CL_dev / IT_dev ID overlap"
assert not ((cltr_ids | cldv_ids) & (ittr_ids | itdv_ids)), "Cross-domain labeled ID overlap"

def position_features(i, n):
    if not USE_POS:
        return [0.0, 0.0, 0.0, 0.0]
    return [i / max(1, n - 1), math.log1p(n) / 8.0, float(i == 0), float(i == n - 1)]

# Flatten into sentence-level records while preserving document context.
def make_labeled_records(docs, domain):
    out = []
    for d in docs:
        sents = d["text"]
        labs = d["labels"]
        did = normalize_id(d["id"])
        for i, (sent, lab) in enumerate(zip(sents, labs)):
            out.append({
                "doc_id": did,
                "sent_idx": i,
                "domain": domain,
                "prev": " ".join(sents[max(0, i-CONTEXT_SENTS):i]) if CONTEXT_SENTS > 0 else "",
                "target": sent,
                "next": " ".join(sents[i+1:i+1+CONTEXT_SENTS]) if CONTEXT_SENTS > 0 else "",
                "pos": position_features(i, len(sents)),
                "label": lab,
                "label_id": LABEL_TO_ID[lab],
            })
    return out

train_records = make_labeled_records(cl_train_raw, "CL") + make_labeled_records(it_train_raw, "IT")
dev_records   = make_labeled_records(cl_dev_raw, "CL") + make_labeled_records(it_dev_raw, "IT")

print("Train sentence records:", len(train_records))
print("Dev sentence records  :", len(dev_records))

train_counts = Counter(x["label"] for x in train_records)
print("\nTrain distribution:")
for lab in LABELS:
    print(f"{lab:30s} {train_counts[lab]:6d}")

# sqrt inverse-frequency; normalize mean to 1; then cap extremes.
N = len(train_records)
C = NUM_LABELS
raw_w = np.array(
    [math.sqrt(N / (C * max(train_counts[lab], 1))) for lab in LABELS],
    dtype=np.float32
)
raw_w = raw_w / raw_w.mean()
class_w = np.clip(raw_w, CLASS_WEIGHT_MIN, CLASS_WEIGHT_MAX)
class_w = torch.tensor(class_w, dtype=torch.float32, device=DEVICE)

print("\nClass weights:")
for lab, w in zip(LABELS, class_w.detach().cpu().tolist()):
    print(f"{lab:30s} {w:.4f}")


Labels:
0 Fact
1 Issue
2 ArgumentPetitioner
3 ArgumentRespondent
4 Statute
5 Dissent
6 PrecedentReliedUpon
7 PrecedentNotReliedUpon
8 PrecedentOverruled
9 RulingByLowerCourt
10 RatioOfTheDecision
11 RulingByPresentCourt
12 None
CL_train: docs=40 sentences=10593
CL_dev: docs=5 sentences=1396
IT_train: docs=40 sentences=6252
IT_dev: docs=5 sentences=746
Train sentence records: 16845
Dev sentence records  : 2142

Train distribution:
Fact                             4098
Issue                             107
ArgumentPetitioner               1613
ArgumentRespondent               1348
Statute                          1099
Dissent                           284
PrecedentReliedUpon              1945
PrecedentNotReliedUpon            569
PrecedentOverruled                  3
RulingByLowerCourt                874
RatioOfTheDecision               4301
RulingByPresentCourt              307
None                              297

Class weights:
Fact                           0.4500
Issue             

In [10]:
# ============================================================
# CELL 9 — Target-preserving context tokenization
# ============================================================
_ENC_CACHE={}
def enc(text):
    hit=_ENC_CACHE.get(text)
    if hit is None:
        hit=tokenizer.encode(text); _ENC_CACHE[text]=hit
    return hit

def crop_head_tail(ids,budget):
    if len(ids)<=budget: return list(ids)
    if budget<=0: return []
    a=budget//2; return list(ids[:a])+list(ids[-(budget-a):])

def build_context_item(domain,prev_text,target_text,next_text,max_len=MAX_LEN):
    # Markers are ordinary text; no tokenizer-vocabulary changes.
    prefix=enc(f"Domain: {domain}\nPrevious context:\n")
    tmark=enc("\nTarget sentence:\n")
    nmark=enc("\nNext context:\n")
    suffix=enc("\nRhetorical role:\n")
    prev=enc(prev_text)[-PREV_BUDGET:] if prev_text else []
    targ=crop_head_tail(enc(target_text),TARGET_BUDGET)
    nxt=enc(next_text)[:NEXT_BUDGET] if next_text else []

    fixed=len(prefix)+len(tmark)+len(nmark)+len(suffix)
    avail=max_len-fixed
    if avail<16: raise RuntimeError("Marker text unexpectedly consumes MAX_LEN")
    tkeep=min(len(targ), max(16,min(TARGET_BUDGET,avail)))
    remain=avail-tkeep
    pkeep=min(len(prev),remain//2); nkeep=min(len(nxt),remain-pkeep)
    spare=remain-pkeep-nkeep
    add=min(spare,len(prev)-pkeep); pkeep+=add; spare-=add
    add=min(spare,len(nxt)-nkeep); nkeep+=add; spare-=add
    prev=prev[-pkeep:] if pkeep else []; targ=crop_head_tail(targ,tkeep); nxt=nxt[:nkeep] if nkeep else []

    ids=prefix+prev+tmark
    target_start=len(ids)
    ids += targ
    target_end=len(ids)
    ids += nmark+nxt+suffix
    if len(ids)>max_len: raise RuntimeError((len(ids),max_len))
    assert target_end>target_start
    return ids,target_start,target_end

lengths=[]; target_lengths=[]
for r in train_records[:1000]+dev_records[:500]:
    ids,a,b=build_context_item(r['domain'],r['prev'],r['target'],r['next'])
    lengths.append(len(ids)); target_lengths.append(b-a)
print("Context-token sample: mean=",round(float(np.mean(lengths)),2),"p95=",round(float(np.percentile(lengths,95)),2),"max=",max(lengths))
print("Target-token sample : mean=",round(float(np.mean(target_lengths)),2),"p95=",round(float(np.percentile(target_lengths,95)),2))


Context-token sample: mean= 199.64 p95= 278.05 max= 384
Target-token sample : mean= 70.06 p95= 144.0


In [11]:
# ============================================================
# CELL 10 — Dataset, collator, and Model2-LORMA RR classifier
# ============================================================
class RRDataset(Dataset):
    """Tokenize each sentence/context record once; store ids compactly as uint16."""
    def __init__(self, records, labeled=True):
        assert VOCAB_SIZE <= 65535
        self.records = records
        self.labeled = labeled
        self.items = []
        self.lengths = []

        for r in records:
            ids, a, b = build_context_item(
                r["domain"], r["prev"], r["target"], r["next"]
            )
            self.items.append((np.asarray(ids, dtype=np.uint16), a, b))
            self.lengths.append(len(ids))

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        r = self.records[idx]
        ids, a, b = self.items[idx]
        item = {
            "input_ids": ids,
            "target_start": a,
            "target_end": b,
            "doc_id": r["doc_id"],
            "sent_idx": r["sent_idx"],
            "domain": r["domain"],
            "pos": r["pos"],
        }
        if self.labeled:
            item["label_id"] = r["label_id"]
        return item


def collate_rr(batch):
    T = max(len(x["input_ids"]) for x in batch)
    B = len(batch)

    ids = torch.full(
        (B, T), tokenizer.pad_token_id, dtype=torch.long
    )
    valid = torch.zeros((B, T), dtype=torch.bool)
    tmask = torch.zeros((B, T), dtype=torch.bool)

    pos = torch.tensor(
        [x["pos"] for x in batch], dtype=torch.float32
    )
    domain_id = torch.tensor(
        [0 if x["domain"]=="CL" else 1 for x in batch],
        dtype=torch.long
    )

    for i, x in enumerate(batch):
        z = torch.from_numpy(x["input_ids"].astype(np.int64))
        L = len(z)
        ids[i, :L] = z
        valid[i, :L] = True
        tmask[i, x["target_start"]:x["target_end"]] = True

    out = {
        "input_ids": ids,
        "key_valid": valid,
        "target_mask": tmask,
        "pos": pos,
        "domain_id": domain_id,
        "doc_id": [x["doc_id"] for x in batch],
        "sent_idx": [x["sent_idx"] for x in batch],
        "domain": [x["domain"] for x in batch],
    }

    if "label_id" in batch[0]:
        out["labels"] = torch.tensor(
            [x["label_id"] for x in batch],
            dtype=torch.long
        )
    return out


HEAD_PREFIXES = (
    "sequence_gru.",
    "head_norm.",
    "pos_mlp.",
    "domain_emb.",
    "fc1.",
    "classifier.",
)

class RRClassifier(nn.Module):
    """
    Frozen causal Model-2 backbone + Q/V LORMA + BiGRU task head.

    Important: the transformer itself stays causal. The BiGRU is a downstream
    classification head and supplies task-specific right-context information.
    """
    def __init__(self, backbone, num_labels=13, dropout=HEAD_DROPOUT):
        super().__init__()
        self.backbone = backbone
        self.sequence_gru = nn.GRU(
            input_size=D_MODEL,
            hidden_size=HEAD_HIDDEN,
            batch_first=True,
            bidirectional=True,
        )
        feat_dim = 2 * HEAD_HIDDEN

        self.head_norm = nn.LayerNorm(2 * feat_dim)
        self.pos_mlp = nn.Sequential(
            nn.Linear(4, 64),
            nn.GELU(),
        )
        self.domain_emb = nn.Embedding(2, 32)
        self.fc1 = nn.Linear(2 * feat_dim + 64 + 32, 384)
        self.drop = nn.Dropout(dropout)
        self.classifier = nn.Linear(384, num_labels)

    def forward(self, input_ids, key_valid, target_mask, pos, domain_id):
        # Exact Phase-1 causal transformer path.
        h, _ = self.backbone.forward_hidden(
            input_ids,
            key_valid=key_valid,
            use_cache=False,
        )

        # Exclude right-padding from the downstream BiGRU.
        lengths = key_valid.sum(1).clamp(min=1).cpu()
        packed = nn.utils.rnn.pack_padded_sequence(
            self.drop(h),
            lengths,
            batch_first=True,
            enforce_sorted=False,
        )
        packed_out, _ = self.sequence_gru(packed)
        h2, _ = nn.utils.rnn.pad_packed_sequence(
            packed_out,
            batch_first=True,
            total_length=h.size(1),
        )
        h2 = h2.float()

        vm = key_valid.unsqueeze(-1).to(h2.dtype)
        tm = target_mask.unsqueeze(-1).to(h2.dtype)

        global_mean = (h2 * vm).sum(1) / vm.sum(1).clamp(min=1)
        target_mean = (h2 * tm).sum(1) / tm.sum(1).clamp(min=1)

        x = self.head_norm(
            torch.cat([target_mean, global_mean], dim=-1)
        )
        z = torch.cat([
            x,
            self.pos_mlp(pos.float()),
            self.domain_emb(domain_id),
        ], dim=-1)

        z = F.gelu(self.fc1(self.drop(z)))
        return self.classifier(self.drop(z))


model = RRClassifier(backbone, NUM_LABELS, HEAD_DROPOUT).to(DEVICE)

# LORMA factors were already marked trainable; explicitly enable only task-head params.
for n,p in model.named_parameters():
    if n.startswith(HEAD_PREFIXES):
        p.requires_grad = True

# Guard against accidental backbone unfreezing.
for n,p in model.backbone.named_parameters():
    if "lorma_" not in n:
        assert not p.requires_grad, f"Unexpected trainable Phase-1 parameter: {n}"

trainable = [(n,p.numel()) for n,p in model.named_parameters() if p.requires_grad]
lorma_params = sum(
    p.numel() for n,p in model.named_parameters()
    if p.requires_grad and "lorma_" in n
)
head_params_count = sum(
    p.numel() for n,p in model.named_parameters()
    if p.requires_grad and n.startswith(HEAD_PREFIXES)
)

print("Total params:", f"{sum(p.numel() for p in model.parameters())/1e6:.3f}M")
print("Trainable params:", f"{sum(v for _,v in trainable)/1e6:.3f}M")
print(
    "Trainable %:",
    f"{100*sum(v for _,v in trainable)/sum(p.numel() for p in model.parameters()):.4f}%"
)
print("LORMA params:", lorma_params)
print("Task-head params:", head_params_count)
print("First trainable names:")
for x in trainable[:18]:
    print(" ", x)


Total params: 81.686M
Trainable params: 1.950M
Trainable %: 2.3876%
LORMA params: 133120
Task-head params: 1817229
First trainable names:
  ('backbone.blocks.0.attn.qkv.lorma_A.q', 2560)
  ('backbone.blocks.0.attn.qkv.lorma_A.v', 2560)
  ('backbone.blocks.0.attn.qkv.lorma_B.q', 2560)
  ('backbone.blocks.0.attn.qkv.lorma_B.v', 2560)
  ('backbone.blocks.1.attn.qkv.lorma_A.q', 2560)
  ('backbone.blocks.1.attn.qkv.lorma_A.v', 2560)
  ('backbone.blocks.1.attn.qkv.lorma_B.q', 2560)
  ('backbone.blocks.1.attn.qkv.lorma_B.v', 2560)
  ('backbone.blocks.2.attn.qkv.lorma_A.q', 2560)
  ('backbone.blocks.2.attn.qkv.lorma_A.v', 2560)
  ('backbone.blocks.2.attn.qkv.lorma_B.q', 2560)
  ('backbone.blocks.2.attn.qkv.lorma_B.v', 2560)
  ('backbone.blocks.3.attn.qkv.lorma_A.q', 2560)
  ('backbone.blocks.3.attn.qkv.lorma_A.v', 2560)
  ('backbone.blocks.3.attn.qkv.lorma_B.q', 2560)
  ('backbone.blocks.3.attn.qkv.lorma_B.v', 2560)
  ('backbone.blocks.4.attn.qkv.lorma_A.q', 2560)
  ('backbone.blocks.4.attn.qk

In [12]:
# ============================================================
# CELL 11 — Dataloaders, optimizer, scheduler
# ============================================================
class SortishBatchSampler(torch.utils.data.Sampler):
    """
    Deterministic per-epoch shuffled batches with reduced padding.
    Every record appears exactly once per epoch.
    """
    def __init__(self, lengths, batch_size, seed=SEED, bucket_mult=LENGTH_BUCKET_MULT):
        self.lengths = list(map(int, lengths))
        self.batch_size = int(batch_size)
        self.seed = int(seed)
        self.bucket_size = max(
            self.batch_size,
            self.batch_size * int(bucket_mult)
        )
        self.epoch = 0

    def set_epoch(self, epoch):
        self.epoch = int(epoch)

    def __len__(self):
        return math.ceil(len(self.lengths) / self.batch_size)

    def __iter__(self):
        rng = np.random.default_rng(self.seed + self.epoch)
        idx = np.arange(len(self.lengths), dtype=np.int64)
        rng.shuffle(idx)

        batches = []
        for s in range(0, len(idx), self.bucket_size):
            chunk = idx[s:s+self.bucket_size]
            chunk = sorted(
                chunk.tolist(),
                key=lambda i:self.lengths[i],
                reverse=True,
            )
            batches.extend(
                chunk[j:j+self.batch_size]
                for j in range(0, len(chunk), self.batch_size)
            )

        rng.shuffle(batches)
        yield from batches


def make_loader(records, labeled, batch_size, shuffle):
    t0 = time.time()
    ds = RRDataset(records, labeled)
    print(f"tokenized {len(ds)} records in {time.time()-t0:.1f}s")

    kw = dict(
        collate_fn=collate_rr,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE.type=="cuda"),
        persistent_workers=NUM_WORKERS>0,
    )

    if shuffle and LENGTH_GROUPED_TRAINING:
        bs = SortishBatchSampler(ds.lengths, batch_size, seed=SEED)
        return DataLoader(ds, batch_sampler=bs, **kw)

    if shuffle:
        g = torch.Generator()
        g.manual_seed(SEED)
        return DataLoader(
            ds,
            batch_size=batch_size,
            shuffle=True,
            generator=g,
            **kw,
        )

    order = sorted(range(len(ds)), key=lambda i:ds.lengths[i])
    return DataLoader(
        Subset(ds, order),
        batch_size=batch_size,
        shuffle=False,
        **kw,
    )


def make_adamw(lorma_params, head_params):
    groups = [
        {"params": lorma_params, "lr": LORMA_LR, "weight_decay": 0.0},
        {"params": head_params, "lr": HEAD_LR, "weight_decay": WEIGHT_DECAY},
    ]

    if DEVICE.type == "cuda":
        try:
            opt = torch.optim.AdamW(groups, fused=True)
            print("AdamW fused: True")
            return opt
        except Exception as e:
            print("Fused AdamW unavailable; standard AdamW:", repr(e)[:120])

    print("AdamW fused: False")
    return torch.optim.AdamW(groups)


train_loader = make_loader(
    train_records, True, MICRO_BATCH_SIZE, True
)
dev_loader = make_loader(
    dev_records, True, EVAL_BATCH_SIZE, False
)

lorma_params = []
head_params = []

for n,p in model.named_parameters():
    if not p.requires_grad:
        continue
    if "lorma_" in n:
        lorma_params.append(p)
    else:
        assert n.startswith(HEAD_PREFIXES), n
        head_params.append(p)

assert lorma_params and head_params

optimizer = make_adamw(lorma_params, head_params)

updates_per_epoch = math.ceil(
    len(train_loader) / GRAD_ACCUM_STEPS
)
total_updates = NUM_EPOCHS * updates_per_epoch
warmup_updates = max(
    1, int(total_updates * WARMUP_RATIO)
)

def lr_mult(step):
    if step < warmup_updates:
        return max(1e-8, step / warmup_updates)
    p = (step-warmup_updates) / max(1, total_updates-warmup_updates)
    return 0.5 * (
        1 + math.cos(math.pi * min(max(p,0.0),1.0))
    )

scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer,
    lr_lambda=lr_mult
)

criterion = nn.CrossEntropyLoss(
    weight=class_w,
    label_smoothing=LABEL_SMOOTHING,
)

amp_enabled = bool(USE_AMP and DEVICE.type=="cuda")
scaler = torch.amp.GradScaler(
    "cuda",
    enabled=amp_enabled,
)

print(
    "Train batches/epoch:", len(train_loader),
    "updates/epoch:", updates_per_epoch,
    "total updates:", total_updates,
    "warmup:", warmup_updates,
    "AMP:", amp_enabled,
)


tokenized 16845 records in 1.5s
tokenized 2142 records in 0.1s
AdamW fused: True
Train batches/epoch: 2106 updates/epoch: 1053 total updates: 6318 warmup: 379 AMP: True


In [13]:
# ============================================================
# CELL 12 — Metrics and evaluation
# ============================================================
def _forward_batch(m,batch):
    return m(batch['input_ids'].to(DEVICE,non_blocking=True),batch['key_valid'].to(DEVICE,non_blocking=True),
             batch['target_mask'].to(DEVICE,non_blocking=True),batch['pos'].to(DEVICE,non_blocking=True),batch['domain_id'].to(DEVICE,non_blocking=True))

@torch.no_grad()
def predict_loader(m,loader):
    m.eval(); yt=[]; yp=[]; meta=[]
    for batch in loader:
        with torch.amp.autocast(device_type='cuda',dtype=torch.float16,enabled=amp_enabled): logits=_forward_batch(m,batch)
        yp.extend(logits.float().argmax(-1).cpu().tolist())
        if 'labels' in batch: yt.extend(batch['labels'].tolist())
        meta.extend(zip(batch['doc_id'],batch['sent_idx'],batch['domain']))
    return yt,yp,meta

def score_ids(y_true,y_pred,title='Combined'):
    ids=list(range(NUM_LABELS)); f=f1_score(y_true,y_pred,labels=ids,average='macro',zero_division=0); p=precision_score(y_true,y_pred,labels=ids,average='macro',zero_division=0); r=recall_score(y_true,y_pred,labels=ids,average='macro',zero_division=0)
    print(f"{title}: F1={f:.6f} P={p:.6f} R={r:.6f}"); return f,p,r

def detailed_dev_eval(m,verbose=True):
    yt,yp,meta=predict_loader(m,dev_loader); combined=score_ids(yt,yp,'Combined CL+IT')
    _pc0=Counter(yp)
    if max(_pc0.values())/max(1,len(yp))>0.60: print('   ⚠️ >60% of predictions are one class — possible collapse:',{LABELS[k]:v for k,v in _pc0.items()})
    for dom in ['CL','IT']:
        idx=[i for i,x in enumerate(meta) if x[2]==dom]; score_ids([yt[i] for i in idx],[yp[i] for i in idx],dom)
    if not verbose: return combined,yt,yp,meta
    present=sorted(set(yt)); absent=[i for i in range(NUM_LABELS) if i not in present]
    print(f"Dev contains {len(present)}/{NUM_LABELS} labels; formal Macro-F1 ceiling from label absence = {len(present)/NUM_LABELS:.6f}")
    if absent: print('Absent dev labels:',[LABELS[i] for i in absent])
    pc,gc=Counter(yp),Counter(yt); fp=f1_score(yt,yp,labels=present,average='macro',zero_division=0)
    print(f"   Macro-F1 over {len(present)} labels present in dev: {fp:.6f}")
    print('   gold counts:',{LABELS[k]:gc[k] for k in present}); print('   pred counts:',{LABELS[k]:pc[k] for k in sorted(pc)})
    print("\nPer-label report (all 13 labels explicitly included):")
    print(classification_report(yt,yp,labels=list(range(NUM_LABELS)),target_names=LABELS,digits=4,zero_division=0))
    return combined,yt,yp,meta

@torch.no_grad()
def predict_logits(m,loader):
    m.eval(); outs=[]; meta=[]; yt=[]
    for batch in loader:
        with torch.amp.autocast(device_type='cuda',dtype=torch.float16,enabled=amp_enabled): lg=_forward_batch(m,batch)
        outs.append(lg.float().cpu().numpy()); meta.extend(zip(batch['doc_id'],batch['sent_idx'],batch['domain']))
        if 'labels' in batch: yt.extend(batch['labels'].tolist())
    return np.concatenate(outs,0),meta,yt


In [14]:
# ============================================================
# CELL 13 — Train Model2 + LORMA; select by combined Macro-F1 (mid-epoch evaluation)
# ============================================================
def fmt_sec(s):
    s=max(0,int(s))
    h,s=divmod(s,3600)
    m,s=divmod(s,60)
    return f"{h:02d}:{m:02d}:{s:02d}"

# Evaluation points inside an epoch, aligned to optimizer-update boundaries.
def make_eval_points(n_batches):
    pts=[]
    for k in range(1,EVAL_PER_EPOCH+1):
        p=int(round(n_batches*k/EVAL_PER_EPOCH/GRAD_ACCUM_STEPS))*GRAD_ACCUM_STEPS
        pts.append(min(n_batches,max(GRAD_ACCUM_STEPS,p)))
    pts[-1]=n_batches
    return set(pts)
EVAL_POINTS = make_eval_points(len(train_loader))
print("Dev evaluation at batches:", sorted(EVAL_POINTS), "of", len(train_loader))

best_f1 = -1.0
best_epoch = None
best_frac = None
best_batch_in_epoch = None
best_global_update = None
bad_evals = 0
global_update = 0
history = []
train_start = time.time()
stop_training = False

optimizer.zero_grad(set_to_none=True)

for epoch in range(1, NUM_EPOCHS+1):
    if hasattr(train_loader.batch_sampler, "set_epoch"):
        train_loader.batch_sampler.set_epoch(epoch)

    model.train()
    running = torch.zeros((), device=DEVICE, dtype=torch.float32)
    n_batches = 0
    ep0 = time.time()

    for step,batch in enumerate(train_loader,1):
        labels = batch["labels"].to(DEVICE, non_blocking=True)

        with torch.amp.autocast(device_type="cuda", dtype=torch.float16, enabled=amp_enabled):
            logits = _forward_batch(model,batch)
            loss = criterion(logits,labels)
            scaled = loss / GRAD_ACCUM_STEPS

        if step % NAN_CHECK_EVERY == 0 or step == len(train_loader):
            if not bool(torch.isfinite(loss.detach()).item()):
                raise FloatingPointError(f"Non-finite loss epoch={epoch} batch={step}")

        scaler.scale(scaled).backward()
        running.add_(loss.detach().float())
        n_batches += 1

        if step % GRAD_ACCUM_STEPS == 0 or step == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], GRAD_CLIP)

            # Keep LR scheduler aligned with actual successful AMP optimizer steps.
            old_scale = scaler.get_scale()
            scaler.step(optimizer)
            scaler.update()
            new_scale = scaler.get_scale()
            optimizer.zero_grad(set_to_none=True)

            if new_scale >= old_scale:
                scheduler.step()
                global_update += 1

        if step % PRINT_EVERY == 0 or step == len(train_loader):
            elapsed = time.time()-ep0
            rate = elapsed / max(step,1)
            eta_ep = rate * (len(train_loader)-step)
            completed_equiv = (epoch-1) + step/len(train_loader)
            eta_total = rate * len(train_loader) * max(0, NUM_EPOCHS-completed_equiv)
            print(
                f"epoch={epoch} batch={step}/{len(train_loader)} "
                f"loss={float((running/max(n_batches,1)).item()):.4f} "
                f"lr_lorma={optimizer.param_groups[0]['lr']:.2e} lr_head={optimizer.param_groups[1]['lr']:.2e} "
                f"elapsed={fmt_sec(elapsed)} ETA_epoch={fmt_sec(eta_ep)} ETA_maxtrain≈{fmt_sec(eta_total)}",
                flush=True,
            )

        if step in EVAL_POINTS:
            frac = (epoch-1) + step/len(train_loader)
            metrics, yt, yp, meta = detailed_dev_eval(model, verbose=False)
            mf, mp, mr = metrics
            _pc = Counter(yp)
            print("   pred counts:", {LABELS[k]:v for k,v in sorted(_pc.items())})
            row = {
                "epoch": epoch,
                "epoch_frac": frac,
                "batch_in_epoch": int(step),
                "global_update": int(global_update),
                "train_loss": float((running/max(n_batches,1)).item()),
                "macro_f1": mf,
                "macro_precision": mp,
                "macro_recall": mr,
                "elapsed_sec": time.time()-ep0,
            }
            history.append(row)
            print("Eval summary:", row)

            if mf > best_f1:
                best_f1 = mf
                best_epoch = int(math.ceil(frac))
                best_frac = float(frac)
                best_batch_in_epoch = int(step)
                best_global_update = int(global_update)
                bad_evals = 0

                task_state = {
                    k:v.detach().cpu()
                    for k,v in model.state_dict().items()
                    if "lorma_" in k or k.startswith(HEAD_PREFIXES)
                }
                torch.save(
                    {
                        "task_state": task_state,
                        "best_epoch": best_epoch,
                        "best_frac": best_frac,
                        "best_batch_in_epoch": best_batch_in_epoch,
                        "best_global_update": best_global_update,
                        "best_macro_f1": best_f1,
                        "history": history,
                        "labels": LABELS,
                        "label_to_id": LABEL_TO_ID,
                        "config": {
                            "experiment": "Phase3 Exp3 Model2 LORMA",
                            "max_len": MAX_LEN,
                            "context_sents": CONTEXT_SENTS,
                            "lorma_variant": "multiplicative_QV_only",
                            "lorma_r": LORMA_R,
                            "lorma_alpha": LORMA_ALPHA,
                            "lorma_dropout": LORMA_DROPOUT,
                            "adapt_q": LORMA_ADAPT_Q,
                            "adapt_k": LORMA_ADAPT_K,
                            "adapt_v": LORMA_ADAPT_V,
                            "backbone_attention": "causal",
                            "head_type": HEAD_TYPE,
                            "head_hidden": HEAD_HIDDEN,
                            "lorma_lr": LORMA_LR,
                            "head_lr": HEAD_LR,
                            "label_smoothing": LABEL_SMOOTHING,
                            "class_weight_min": CLASS_WEIGHT_MIN,
                            "class_weight_max": CLASS_WEIGHT_MAX,
                            "eval_per_epoch": EVAL_PER_EPOCH,
                        },
                    },
                    BEST_CKPT,
                )
                print(f"✅ NEW BEST at epoch {frac:.3f} | batch={step} | update={global_update}:", BEST_CKPT)
            else:
                bad_evals += 1

            print(f"Best so far: epoch_frac={best_frac:.3f} | batch={best_batch_in_epoch} | update={best_global_update} | Macro-F1={best_f1:.6f} | non-improving evals={bad_evals}\n")
            model.train()
            if bad_evals >= EARLY_STOP_PATIENCE:
                print(f"⏹️ Early stopping: no dev improvement for {EARLY_STOP_PATIENCE} evaluations.")
                stop_training = True
                break

    if stop_training:
        break

with open(ROOT/"history.json","w",encoding="utf-8") as f:
    json.dump(history,f,indent=2)

print("Saved:", ROOT/"history.json")
print("Training complete | elapsed:", fmt_sec(time.time()-train_start))
assert best_frac is not None and best_batch_in_epoch is not None and best_global_update is not None
print("BEST_EPOCH =", best_epoch, "| BEST_EPOCH_FRAC =", best_frac, "| BEST_BATCH =", best_batch_in_epoch, "| BEST_UPDATE =", best_global_update, "| BEST_MACRO_F1 =", best_f1)


Dev evaluation at batches: [1052, 2106] of 2106
epoch=1 batch=250/2106 loss=2.1685 lr_lorma=6.60e-05 lr_head=1.98e-04 elapsed=00:00:20 ETA_epoch=00:02:31 ETA_maxtrain≈00:16:54
epoch=1 batch=500/2106 loss=2.0088 lr_lorma=1.32e-04 lr_head=3.96e-04 elapsed=00:00:38 ETA_epoch=00:02:02 ETA_maxtrain≈00:15:28
epoch=1 batch=750/2106 loss=1.8977 lr_lorma=1.97e-04 lr_head=5.92e-04 elapsed=00:00:56 ETA_epoch=00:01:42 ETA_maxtrain≈00:14:57
epoch=1 batch=1000/2106 loss=1.8227 lr_lorma=2.00e-04 lr_head=5.99e-04 elapsed=00:01:14 ETA_epoch=00:01:22 ETA_maxtrain≈00:14:29
Combined CL+IT: F1=0.331747 P=0.409196 R=0.320377
CL: F1=0.239000 P=0.258904 R=0.260676
IT: F1=0.355129 P=0.410608 R=0.339474
   pred counts: {'Fact': 535, 'Issue': 7, 'ArgumentPetitioner': 189, 'ArgumentRespondent': 60, 'Statute': 103, 'Dissent': 13, 'PrecedentReliedUpon': 270, 'PrecedentNotReliedUpon': 20, 'RulingByLowerCourt': 25, 'RatioOfTheDecision': 846, 'RulingByPresentCourt': 41, 'None': 33}
Eval summary: {'epoch': 1, 'epoch_fr

In [15]:
# ============================================================
# CELL 14 — Reload BEST checkpoint and reconstruct dev prediction.json
# ============================================================
best_obj=torch.load(BEST_CKPT,map_location='cpu',weights_only=False)
missing,unexpected=model.load_state_dict(best_obj['task_state'],strict=False)
assert not unexpected,unexpected
model.to(DEVICE).eval(); metrics,y_true,y_pred,meta=detailed_dev_eval(model); reloaded_f1=metrics[0]
print('Stored best F1:',best_obj['best_macro_f1'],'Reloaded F1:',reloaded_f1,'Delta:',abs(best_obj['best_macro_f1']-reloaded_f1))
assert abs(best_obj['best_macro_f1']-reloaded_f1)<5e-3

def records_to_prediction(meta,pred_ids):
    temp={}
    for (did,si,dom),pid in zip(meta,pred_ids): temp.setdefault(did,[]).append((si,ID_TO_LABEL[int(pid)]))
    return {did:[lab for _,lab in sorted(pairs)] for did,pairs in temp.items()}
dev_prediction=records_to_prediction(meta,y_pred)
with open(DEV_PRED,'w',encoding='utf-8') as f: json.dump(dev_prediction,f,ensure_ascii=False,indent=2)
def truth_map(docs): return {normalize_id(d['id']):list(d['labels']) for d in docs}
with open(CL_DEV_TRUTH,'w',encoding='utf-8') as f: json.dump(truth_map(cl_dev_raw),f,ensure_ascii=False,indent=2)
with open(IT_DEV_TRUTH,'w',encoding='utf-8') as f: json.dump(truth_map(it_dev_raw),f,ensure_ascii=False,indent=2)
with open(LABELS_JSON,'w',encoding='utf-8') as f: json.dump(label_vocab_raw,f,ensure_ascii=False,indent=2)
print('Saved:',CL_DEV_TRUTH,IT_DEV_TRUTH,DEV_PRED,LABELS_JSON,sep='\n')


Combined CL+IT: F1=0.368528 P=0.438824 R=0.350285
CL: F1=0.293268 P=0.329021 R=0.298725
IT: F1=0.375384 P=0.464502 R=0.350976
Dev contains 11/13 labels; formal Macro-F1 ceiling from label absence = 0.846154
Absent dev labels: ['Dissent', 'PrecedentOverruled']
   Macro-F1 over 11 labels present in dev: 0.435533
   gold counts: {'Fact': 444, 'Issue': 15, 'ArgumentPetitioner': 400, 'ArgumentRespondent': 272, 'Statute': 120, 'PrecedentReliedUpon': 254, 'PrecedentNotReliedUpon': 50, 'RulingByLowerCourt': 102, 'RatioOfTheDecision': 403, 'RulingByPresentCourt': 49, 'None': 33}
   pred counts: {'Fact': 700, 'Issue': 10, 'ArgumentPetitioner': 221, 'ArgumentRespondent': 107, 'Statute': 133, 'Dissent': 9, 'PrecedentReliedUpon': 155, 'PrecedentNotReliedUpon': 51, 'RulingByLowerCourt': 52, 'RatioOfTheDecision': 661, 'RulingByPresentCourt': 29, 'None': 14}

Per-label report (all 13 labels explicitly included):
                        precision    recall  f1-score   support

                  Fact   

In [17]:
# ============================================================
# CELL 15 — EXACT local Phase-3 evaluator script supplied by course
# ============================================================

EVALUATOR_CODE = r'''#!/usr/bin/env python3
"""Local evaluator for Phase 3.

The competition scorer evaluates two gold/reference JSON files together and
one participant prediction.json. This local evaluator mirrors that setup
using the student's own held-out splits.
"""
import argparse
import json
import sys

from sklearn.metrics import f1_score, precision_score, recall_score

EXPECTED_LABEL_COUNT = 13


def load_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def normalize_id(doc_id):
    if not isinstance(doc_id, str) or not doc_id.strip():
        raise ValueError("Document IDs must be non-empty strings.")
    return doc_id[:-4] if doc_id.endswith(".txt") else doc_id


def load_vocab(data):
    if isinstance(data, list):
        labels = data
    elif isinstance(data, dict) and data:
        values = list(data.values())
        if all(isinstance(v, int) and not isinstance(v, bool) for v in values):
            labels = list(data.keys())
        elif all(isinstance(v, str) for v in values):
            labels = values
        else:
            raise ValueError("Unsupported labels.json format.")
    else:
        raise ValueError("labels.json must be a non-empty list or object.")

    if len(labels) != EXPECTED_LABEL_COUNT:
        raise ValueError(
            f"Expected {EXPECTED_LABEL_COUNT} labels, found {len(labels)}."
        )
    if any(not isinstance(label, str) or not label.strip() for label in labels):
        raise ValueError("Labels must be unique non-empty strings.")
    if len(set(labels)) != len(labels):
        raise ValueError("Labels must be unique non-empty strings.")
    return labels


def normalize_reference(data, source_name):
    if not isinstance(data, dict):
        raise ValueError(f"{source_name}: reference data must be a JSON object.")

    normalized = {}
    for raw_id, labels in data.items():
        doc_id = normalize_id(raw_id)
        if doc_id in normalized:
            raise ValueError(
                f"{source_name}: duplicate document ID after .txt normalization: {doc_id}"
            )
        normalized[doc_id] = labels
    return normalized


def normalize_predictions(data):
    if not isinstance(data, dict):
        raise ValueError("prediction.json must be a JSON object mapping document IDs to label lists.")

    normalized = {}
    for raw_id, labels in data.items():
        doc_id = normalize_id(raw_id)
        if doc_id in normalized:
            raise ValueError(
                f"prediction.json: duplicate document ID after .txt normalization: {doc_id}"
            )
        normalized[doc_id] = labels
    return normalized


def main():
    parser = argparse.ArgumentParser(
        description=(
            "Evaluate Phase 3 sentence-label predictions on two local held-out "
            "reference splits, mirroring the competition's combined scoring setup."
        )
    )
    parser.add_argument("--truth-cl", required=True)
    parser.add_argument("--truth-it", required=True)
    parser.add_argument("--prediction", required=True)
    parser.add_argument("--labels", required=True)
    args = parser.parse_args()

    truth_cl = normalize_reference(load_json(args.truth_cl), "--truth-cl")
    truth_it = normalize_reference(load_json(args.truth_it), "--truth-it")
    predictions = normalize_predictions(load_json(args.prediction))
    labels = load_vocab(load_json(args.labels))
    allowed = set(labels)

    overlap = set(truth_cl) & set(truth_it)
    if overlap:
        sample = sorted(overlap)[:5]
        suffix = "..." if len(overlap) > 5 else ""
        raise ValueError(
            "The two reference splits contain overlapping document IDs: "
            + ", ".join(sample)
            + suffix
        )

    truth = {}
    truth.update(truth_cl)
    truth.update(truth_it)

    if set(truth) != set(predictions):
        missing = sorted(set(truth) - set(predictions))
        extra = sorted(set(predictions) - set(truth))
        parts = [f"missing={len(missing)}", f"extra={len(extra)}"]
        if missing:
            parts.append(f"example missing ID(s)={missing[:5]}")
        if extra:
            parts.append(f"example extra ID(s)={extra[:5]}")
        raise ValueError("ID mismatch: " + "; ".join(parts))

    y_true = []
    y_pred = []

    for doc_id in truth:
        gold = truth[doc_id]
        pred = predictions[doc_id]

        if not isinstance(gold, list) or not isinstance(pred, list):
            raise ValueError(f"{doc_id}: labels must be JSON lists.")
        if len(gold) != len(pred):
            raise ValueError(
                f"{doc_id}: sentence-count mismatch ({len(gold)} vs {len(pred)})."
            )
        if any(label not in allowed for label in gold + pred):
            raise ValueError(f"{doc_id}: unknown label encountered.")

        y_true.extend(gold)
        y_pred.extend(pred)

    score = f1_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)
    macro_precision = precision_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)
    macro_recall = recall_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)

    print(f"CL documents: {len(truth_cl)}")
    print(f"IT documents: {len(truth_it)}")
    print(f"Total documents: {len(truth)}")
    print(f"Total sentences: {len(y_true)}")
    print(f"Macro-F1:        {score:.6f}")
    print(f"Macro-Precision: {macro_precision:.6f}")
    print(f"Macro-Recall:    {macro_recall:.6f}")


if __name__ == "__main__":
    try:
        main()
    except Exception as exc:
        print(f"EVALUATION FAILED: {exc}", file=sys.stderr)
        sys.exit(1)
'''

with open(EVAL_SCRIPT, "w", encoding="utf-8") as f:
    f.write(EVALUATOR_CODE)

print("Saved exact evaluator to:", EVAL_SCRIPT)


Saved exact evaluator to: /kaggle/working/phase3_exp3_model2_lorma/evaluate_phase3.py


In [18]:
# ============================================================
# CELL 16 — RUN THE OFFICIAL-STYLE LOCAL EVALUATOR
# Do not continue to final refit until this passes.
# ============================================================

cmd = [
    sys.executable,
    str(EVAL_SCRIPT),
    "--truth-cl", str(CL_DEV_TRUTH),
    "--truth-it", str(IT_DEV_TRUTH),
    "--prediction", str(DEV_PRED),
    "--labels", str(LABELS_JSON),
]

print("Running:")
print(" ".join(cmd))

result = subprocess.run(
    cmd,
    capture_output=True,
    text=True
)

print("\nSTDOUT:\n", result.stdout)
if result.stderr:
    print("\nSTDERR:\n", result.stderr)

assert result.returncode == 0, "❌ Phase-3 evaluator failed."
print("✅ DEV prediction passed the exact Phase-3 evaluator.")


Running:
/usr/bin/python3 /kaggle/working/phase3_exp3_model2_lorma/evaluate_phase3.py --truth-cl /kaggle/working/phase3_exp3_model2_lorma/cl_dev_truth.json --truth-it /kaggle/working/phase3_exp3_model2_lorma/it_dev_truth.json --prediction /kaggle/working/phase3_exp3_model2_lorma/prediction_dev.json --labels /kaggle/working/phase3_exp3_model2_lorma/labels.json

STDOUT:
 CL documents: 5
IT documents: 5
Total documents: 10
Total sentences: 2142
Macro-F1:        0.368528
Macro-Precision: 0.438824
Macro-Recall:    0.350285

✅ DEV prediction passed the exact Phase-3 evaluator.


In [19]:
# ============================================================
# CELL 17 — Optional transition-aware decoding tuned on DEV
#
# Improvements:
#   * CL and IT transition/prior statistics estimated separately
#   * additive smoothing
#   * no hard "impossible label" bans
#   * only adopt decoding when DEV Macro-F1 improves by >= 0.005
# ============================================================

from sklearn.metrics import f1_score as _f1

def transition_matrix(docs_list, smooth=1.0):
    T = np.full((NUM_LABELS, NUM_LABELS), float(smooth), dtype=np.float64)
    for d in docs_list:
        ids = [LABEL_TO_ID[x] for x in d["labels"]]
        for a, b in zip(ids[:-1], ids[1:]):
            T[a, b] += 1.0
    T /= T.sum(1, keepdims=True)
    return np.log(T)

def class_log_prior(docs_list, smooth=1.0):
    c = np.full(NUM_LABELS, float(smooth), dtype=np.float64)
    for d in docs_list:
        for x in d["labels"]:
            c[LABEL_TO_ID[x]] += 1.0
    return np.log(c / c.sum())

def log_softmax_np(x):
    x = x - x.max(-1, keepdims=True)
    return x - np.log(np.exp(x).sum(-1, keepdims=True))

def viterbi_decode(logp, trans, beta):
    n, K = logp.shape
    score = logp[0].copy()
    back = np.zeros((n, K), dtype=np.int32)
    tr = beta * trans

    for t in range(1, n):
        cand = score[:, None] + tr
        back[t] = cand.argmax(0)
        score = cand[back[t], np.arange(K)] + logp[t]

    path = np.zeros(n, dtype=np.int64)
    path[-1] = int(score.argmax())

    for t in range(n - 1, 0, -1):
        path[t - 1] = back[t, path[t]]

    return path

def build_domain_stats(cl_docs, it_docs):
    return {
        "CL": {
            "trans": transition_matrix(cl_docs, smooth=1.0),
            "prior": class_log_prior(cl_docs, smooth=1.0),
        },
        "IT": {
            "trans": transition_matrix(it_docs, smooth=1.0),
            "prior": class_log_prior(it_docs, smooth=1.0),
        },
    }

def decode_docs(logits, meta, domain_stats, alpha, beta):
    # logits aligned with meta=(doc_id, sent_idx, domain)
    by_doc = {}
    for i, (did, si, dom) in enumerate(meta):
        by_doc.setdefault(did, []).append((si, i, dom))

    pred = np.zeros(len(meta), dtype=np.int64)

    for did, items in by_doc.items():
        items.sort(key=lambda x: x[0])
        idx = [i for _, i, _ in items]
        dom = items[0][2]
        stats = domain_stats[dom]

        lp = (
            log_softmax_np(logits[idx].astype(np.float64))
            - alpha * stats["prior"][None, :]
        )

        pred[idx] = (
            viterbi_decode(lp, stats["trans"], beta)
            if beta > 0
            else lp.argmax(-1)
        )

    return pred

# Tune only on held-out DEV predictions.
dev_logits, dev_meta, dev_y = predict_logits(model, dev_loader)
np.savez_compressed(
    ROOT / "dev_logits.npz",
    logits=dev_logits,
    y=np.asarray(dev_y, dtype=np.int64),
    doc_id=np.asarray([x[0] for x in dev_meta]),
    sent_idx=np.asarray([x[1] for x in dev_meta], dtype=np.int64),
    domain=np.asarray([x[2] for x in dev_meta]),
)

domain_stats_train = build_domain_stats(
    cl_train_raw,
    it_train_raw,
)

def dev_f1(alpha, beta):
    p = decode_docs(
        dev_logits,
        dev_meta,
        domain_stats_train,
        alpha,
        beta
    )
    return _f1(
        dev_y,
        p,
        labels=list(range(NUM_LABELS)),
        average="macro",
        zero_division=0
    )

rows = []
for alpha in [-0.25, 0.0, 0.25, 0.5, 0.75]:
    for beta in [0.0, 0.25, 0.5, 1.0]:
        rows.append((dev_f1(alpha, beta), alpha, beta))

base_f1 = dev_f1(0.0, 0.0)
rows.sort(key=lambda r: (-r[0], r[2], abs(r[1])))

print(f"Baseline argmax (alpha=0, beta=0): {base_f1:.6f}")
print("Top decoding settings (dev F1, alpha, beta):")
for r in rows[:8]:
    print(f"  {r[0]:.6f}  alpha={r[1]:+.2f}  beta={r[2]:.2f}")

if rows[0][0] - base_f1 >= 0.005:
    DECODE_ALPHA, DECODE_BETA = rows[0][1], rows[0][2]
else:
    DECODE_ALPHA, DECODE_BETA = 0.0, 0.0

print("SELECTED decoding: alpha=", DECODE_ALPHA, "beta=", DECODE_BETA)

dec_ids = decode_docs(
    dev_logits,
    dev_meta,
    domain_stats_train,
    DECODE_ALPHA,
    DECODE_BETA
)

dev_pred_dec = records_to_prediction(dev_meta, dec_ids)
DEV_PRED_DEC = ROOT / "prediction_dev_decoded.json"

with open(DEV_PRED_DEC, "w", encoding="utf-8") as f:
    json.dump(dev_pred_dec, f, ensure_ascii=False, indent=2)

# Run the exact supplied evaluator again on the decoded DEV prediction.
res = subprocess.run(
    [
        sys.executable,
        str(EVAL_SCRIPT),
        "--truth-cl", str(CL_DEV_TRUTH),
        "--truth-it", str(IT_DEV_TRUTH),
        "--prediction", str(DEV_PRED_DEC),
        "--labels", str(LABELS_JSON),
    ],
    capture_output=True,
    text=True
)

print(res.stdout)
if res.stderr:
    print(res.stderr)

assert res.returncode == 0, "Decoded DEV prediction failed the exact evaluator."
print("✅ Decoded DEV prediction passed the exact Phase-3 evaluator.")


Baseline argmax (alpha=0, beta=0): 0.368528
Top decoding settings (dev F1, alpha, beta):
  0.392478  alpha=+0.25  beta=0.25
  0.386194  alpha=+0.50  beta=0.50
  0.378508  alpha=+0.50  beta=1.00
  0.376910  alpha=+0.50  beta=0.25
  0.376662  alpha=+0.25  beta=0.50
  0.369838  alpha=+0.00  beta=0.25
  0.369654  alpha=+0.00  beta=0.50
  0.368554  alpha=-0.25  beta=0.00
SELECTED decoding: alpha= 0.25 beta= 0.25
CL documents: 5
IT documents: 5
Total documents: 10
Total sentences: 2142
Macro-F1:        0.392478
Macro-Precision: 0.465848
Macro-Recall:    0.376381

✅ Decoded DEV prediction passed the exact Phase-3 evaluator.


In [20]:
# ============================================================
# CELL 18 — Fresh Phase-1 Model2 + LORMA for train+dev refit
# ============================================================
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

BEST_EPOCH = int(best_obj["best_epoch"])
BEST_FRAC = float(best_obj.get("best_frac", BEST_EPOCH))
BEST_BATCH = int(best_obj.get("best_batch_in_epoch", 0))
BEST_UPDATE = int(best_obj.get("best_global_update", 0))
assert BEST_EPOCH >= 1
assert 0.0 < BEST_FRAC <= NUM_EPOCHS
print(
    "Final refit target selected from dev:",
    f"epoch_frac={BEST_FRAC:.3f}",
    f"| selected_batch={BEST_BATCH}",
    f"| selected_update={BEST_UPDATE}",
)

all_labeled_records = (
    make_labeled_records(cl_train_raw,"CL")
    + make_labeled_records(it_train_raw,"IT")
    + make_labeled_records(cl_dev_raw,"CL")
    + make_labeled_records(it_dev_raw,"IT")
)

all_counts = Counter(
    x["label"] for x in all_labeled_records
)
N_all = len(all_labeled_records)

raw = np.array(
    [
        math.sqrt(
            N_all / (
                NUM_LABELS * max(all_counts[lab],1)
            )
        )
        for lab in LABELS
    ],
    dtype=np.float32,
)
raw /= raw.mean()

class_w_all = torch.tensor(
    np.clip(
        raw,
        CLASS_WEIGHT_MIN,
        CLASS_WEIGHT_MAX,
    ),
    dtype=torch.float32,
    device=DEVICE,
)

def reset_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def build_fresh_rr_from_phase1():
    reset_seed(SEED)

    bb = build_backbone()
    o = torch.load(
        PHASE1_CKPT,
        map_location="cpu",
        weights_only=False,
    )
    st = o["model"] if isinstance(o,dict) and "model" in o else o
    bb.load_state_dict(st, strict=True)

    inject_lorma(bb)

    m = RRClassifier(
        bb,
        NUM_LABELS,
        HEAD_DROPOUT,
    )

    for n,p in m.named_parameters():
        if n.startswith(HEAD_PREFIXES):
            p.requires_grad=True

    for n,p in m.backbone.named_parameters():
        if "lorma_" not in n:
            assert not p.requires_grad, n

    return m.to(DEVICE)

final_model = build_fresh_rr_from_phase1()

all_loader = make_loader(
    all_labeled_records,
    True,
    MICRO_BATCH_SIZE,
    True,
)

# Replay the same *training-position fraction* on the larger train+dev loader.
# Example: BEST_FRAC=1.5 means one full refit epoch + half of the next.
REFIT_TOTAL_BATCHES = int(
    round(BEST_FRAC * len(all_loader) / GRAD_ACCUM_STEPS)
) * GRAD_ACCUM_STEPS
REFIT_TOTAL_BATCHES = max(GRAD_ACCUM_STEPS, REFIT_TOTAL_BATCHES)

REFIT_FULL_EPOCHS = REFIT_TOTAL_BATCHES // len(all_loader)
REFIT_REMAINDER_BATCHES = REFIT_TOTAL_BATCHES % len(all_loader)

print(
    "Refit target:",
    REFIT_TOTAL_BATCHES,
    "micro-batches total | loader batches/epoch:",
    len(all_loader),
    "| full epochs:",
    REFIT_FULL_EPOCHS,
    "| remainder batches:",
    REFIT_REMAINDER_BATCHES,
)

final_lorma = []
final_head = []

for n,p in final_model.named_parameters():
    if not p.requires_grad:
        continue
    if "lorma_" in n:
        final_lorma.append(p)
    else:
        assert n.startswith(HEAD_PREFIXES), n
        final_head.append(p)

final_optimizer = make_adamw(
    final_lorma,
    final_head,
)

final_updates_per_epoch = math.ceil(
    len(all_loader)/GRAD_ACCUM_STEPS
)

SCHED_EPOCHS = (
    NUM_EPOCHS
    if REFIT_MATCH_ORIGINAL_SCHEDULE
    else BEST_EPOCH
)

final_total_updates = (
    SCHED_EPOCHS * final_updates_per_epoch
)
final_warmup = max(
    1,
    int(final_total_updates * WARMUP_RATIO),
)

def final_lr_mult(step):
    if step < final_warmup:
        return max(1e-8, step/final_warmup)

    p = (
        (step-final_warmup)
        / max(1,final_total_updates-final_warmup)
    )
    return 0.5 * (
        1 + math.cos(
            math.pi * min(max(p,0.0),1.0)
        )
    )

final_scheduler = torch.optim.lr_scheduler.LambdaLR(
    final_optimizer,
    lr_lambda=final_lr_mult,
)

final_criterion = nn.CrossEntropyLoss(
    weight=class_w_all,
    label_smoothing=LABEL_SMOOTHING,
)

final_scaler = torch.amp.GradScaler(
    "cuda",
    enabled=amp_enabled,
)

print(
    "All labeled records:", len(all_labeled_records),
    "Final batches/epoch:", len(all_loader),
    "Schedule length (updates):", final_total_updates,
    "| updates actually run:",
    math.ceil(REFIT_TOTAL_BATCHES/GRAD_ACCUM_STEPS),
)


Final refit target selected from dev: epoch_frac=2.500 | selected_batch=1052 | selected_update=2630
Injected fused-QKV LORMA modules: 13
Trainable LORMA tensors: 52
LORMA trainable params: 133120
Expected LORMA params: 133120
tokenized 18987 records in 0.3s
Refit target: 5934 micro-batches total | loader batches/epoch: 2374 | full epochs: 2 | remainder batches: 1186
AdamW fused: True
All labeled records: 18987 Final batches/epoch: 2374 Schedule length (updates): 7122 | updates actually run: 2967


In [21]:
# ============================================================
# CELL 19 — Final LORMA refit with live ETA (stops at the selected training position)
# ============================================================
final_optimizer.zero_grad(set_to_none=True)
refit_start = time.time()
refit_updates = 0
batches_done = 0
refit_stop = False

for epoch in range(1, int(math.ceil(REFIT_TOTAL_BATCHES / len(all_loader))) + 1):
    if hasattr(all_loader.batch_sampler, "set_epoch"):
        all_loader.batch_sampler.set_epoch(epoch)

    final_model.train()
    running = torch.zeros((), device=DEVICE, dtype=torch.float32)
    n_batches = 0
    ep0 = time.time()

    for step,batch in enumerate(all_loader,1):
        labels = batch["labels"].to(DEVICE, non_blocking=True)

        with torch.amp.autocast(device_type="cuda", dtype=torch.float16, enabled=amp_enabled):
            logits = _forward_batch(final_model, batch)
            loss = final_criterion(logits, labels)
            scaled = loss / GRAD_ACCUM_STEPS

        if step % NAN_CHECK_EVERY == 0 or step == len(all_loader):
            if not bool(torch.isfinite(loss.detach()).item()):
                raise FloatingPointError(f"Non-finite refit loss epoch={epoch} batch={step}")

        final_scaler.scale(scaled).backward()
        running.add_(loss.detach().float())
        n_batches += 1
        batches_done += 1

        if step % GRAD_ACCUM_STEPS == 0 or step == len(all_loader) or batches_done >= REFIT_TOTAL_BATCHES:
            final_scaler.unscale_(final_optimizer)
            torch.nn.utils.clip_grad_norm_([p for p in final_model.parameters() if p.requires_grad], GRAD_CLIP)

            old_scale = final_scaler.get_scale()
            final_scaler.step(final_optimizer)
            final_scaler.update()
            new_scale = final_scaler.get_scale()
            final_optimizer.zero_grad(set_to_none=True)

            if new_scale >= old_scale:
                final_scheduler.step()
                refit_updates += 1

        if step % PRINT_EVERY == 0 or batches_done >= REFIT_TOTAL_BATCHES:
            elapsed = time.time()-refit_start
            rate = elapsed/max(batches_done,1)
            print(
                f"refit epoch={epoch} batch={step}/{len(all_loader)} done={batches_done}/{REFIT_TOTAL_BATCHES} "
                f"loss={float((running/max(n_batches,1)).item()):.4f} "
                f"elapsed={fmt_sec(elapsed)} ETA_refit≈{fmt_sec(rate*(REFIT_TOTAL_BATCHES-batches_done))}",
                flush=True,
            )

        if batches_done >= REFIT_TOTAL_BATCHES:
            refit_stop = True
            break

    print(f"Refit epoch {epoch} segment complete | loss={float((running/max(n_batches,1)).item()):.4f} | elapsed={fmt_sec(time.time()-ep0)}")
    if refit_stop:
        break

final_task_state = {
    k:v.detach().cpu()
    for k,v in final_model.state_dict().items()
    if "lorma_" in k or k.startswith(HEAD_PREFIXES)
}

torch.save(
    {
        "task_state": final_task_state,
        "refit_epochs_ceiling": BEST_EPOCH,
        "source_best_epoch_frac": BEST_FRAC,
        "source_best_batch_in_epoch": BEST_BATCH,
        "source_best_global_update": BEST_UPDATE,
        "refit_total_batches": REFIT_TOTAL_BATCHES,
        "refit_optimizer_updates": refit_updates,
        "source_best_dev_f1": best_f1,
        "labels": LABELS,
        "label_to_id": LABEL_TO_ID,
        "config": {
            "experiment": "Phase3 Exp3 Model2 LORMA final refit",
            "lorma_r": LORMA_R,
            "lorma_alpha": LORMA_ALPHA,
            "adapt_q": LORMA_ADAPT_Q,
            "adapt_k": LORMA_ADAPT_K,
            "adapt_v": LORMA_ADAPT_V,
            "head_type": HEAD_TYPE,
            "head_hidden": HEAD_HIDDEN,
        },
    },
    FINAL_CKPT,
)

print("✅ Final refit saved:", FINAL_CKPT, "| optimizer updates:", refit_updates, "| total refit:", fmt_sec(time.time()-refit_start))


refit epoch=1 batch=250/2374 done=250/5934 loss=2.2176 elapsed=00:00:18 ETA_refit≈00:07:04
refit epoch=1 batch=500/2374 done=500/5934 loss=2.0225 elapsed=00:00:38 ETA_refit≈00:06:55
refit epoch=1 batch=750/2374 done=750/5934 loss=1.9109 elapsed=00:00:56 ETA_refit≈00:06:27
refit epoch=1 batch=1000/2374 done=1000/5934 loss=1.8481 elapsed=00:01:15 ETA_refit≈00:06:11
refit epoch=1 batch=1250/2374 done=1250/5934 loss=1.7988 elapsed=00:01:34 ETA_refit≈00:05:53
refit epoch=1 batch=1500/2374 done=1500/5934 loss=1.7548 elapsed=00:01:53 ETA_refit≈00:05:35
refit epoch=1 batch=1750/2374 done=1750/5934 loss=1.7061 elapsed=00:02:11 ETA_refit≈00:05:15
refit epoch=1 batch=2000/2374 done=2000/5934 loss=1.6725 elapsed=00:02:30 ETA_refit≈00:04:56
refit epoch=1 batch=2250/2374 done=2250/5934 loss=1.6427 elapsed=00:02:49 ETA_refit≈00:04:37
Refit epoch 1 segment complete | loss=1.6292 | elapsed=00:02:58
refit epoch=2 batch=250/2374 done=2624/5934 loss=1.2894 elapsed=00:03:17 ETA_refit≈00:04:08
refit epoch=2

In [22]:
# ============================================================
# CELL 20 — Build test sentence records
# ============================================================

def make_test_records(test_dict, domain):
    records = []
    for raw_id, sents in test_dict.items():
        did = normalize_id(raw_id)
        assert isinstance(sents, list)
        for i, sent in enumerate(sents):
            records.append({
                "doc_id": did,
                "sent_idx": i,
                "domain": domain,
                "prev": " ".join(sents[max(0, i-CONTEXT_SENTS):i]) if CONTEXT_SENTS > 0 else "",
                "target": sent,
                "next": " ".join(sents[i+1:i+1+CONTEXT_SENTS]) if CONTEXT_SENTS > 0 else "",
                "pos": position_features(i, len(sents)),
            })
    return records

test_records = (
    make_test_records(cl_test_raw, "CL")
    + make_test_records(it_test_raw, "IT")
)

test_loader = make_loader(test_records, False, EVAL_BATCH_SIZE, False)

print("Total test sentence records:", len(test_records))
assert len(test_records) == (
    sum(len(v) for v in cl_test_raw.values())
    + sum(len(v) for v in it_test_raw.values())
)


tokenized 2197 records in 0.2s
Total test sentence records: 2197


In [23]:
# ============================================================
# CELL 21 — Test inference + selected transition decoding
# Variants: heldout (best dev ckpt), refit (train+dev), ensemble (mean log-prob)
# ============================================================
domain_stats_all=build_domain_stats(cl_train_raw+cl_dev_raw,it_train_raw+it_dev_raw)
def _logp(lg): return log_softmax_np(lg.astype(np.float64))
def decode_to_prediction(lg,meta,name):
    ids=decode_docs(lg,meta,domain_stats_all,DECODE_ALPHA,DECODE_BETA); pred=records_to_prediction(meta,ids); cnt=Counter(ids.tolist())
    print(f"[{name}] predicted label counts:",{LABELS[k]:cnt[k] for k in sorted(cnt)}); return pred

lg_held,meta_held,_=predict_logits(model,test_loader)
lg_refit,meta_refit,_=predict_logits(final_model,test_loader)
assert meta_held==meta_refit
np.savez_compressed(
    ROOT/'test_logits.npz',
    heldout=lg_held,
    refit=lg_refit,
    doc_id=np.asarray([x[0] for x in meta_held]),
    sent_idx=np.asarray([x[1] for x in meta_held],dtype=np.int64),
    domain=np.asarray([x[2] for x in meta_held]),
)

predictions={'heldout':decode_to_prediction(lg_held,meta_held,'heldout'),
             'refit':decode_to_prediction(lg_refit,meta_refit,'refit')}
if ENSEMBLE_HELDOUT_AND_REFIT:
    predictions['ensemble']=decode_to_prediction(0.5*(_logp(lg_held)+_logp(lg_refit)),meta_held,'ensemble')
prediction=predictions['refit']
with open(OFFICIAL_PRED,'w',encoding='utf-8') as f: json.dump(prediction,f,ensure_ascii=False,indent=2)
print('Saved:',OFFICIAL_PRED,'Prediction documents:',len(prediction))


[heldout] predicted label counts: {'Fact': 556, 'Issue': 18, 'ArgumentPetitioner': 211, 'ArgumentRespondent': 17, 'Statute': 195, 'Dissent': 9, 'PrecedentReliedUpon': 279, 'PrecedentNotReliedUpon': 37, 'RulingByLowerCourt': 98, 'RatioOfTheDecision': 737, 'RulingByPresentCourt': 24, 'None': 16}
[refit] predicted label counts: {'Fact': 396, 'Issue': 14, 'ArgumentPetitioner': 306, 'ArgumentRespondent': 11, 'Statute': 162, 'Dissent': 3, 'PrecedentReliedUpon': 578, 'PrecedentNotReliedUpon': 27, 'RulingByLowerCourt': 212, 'RatioOfTheDecision': 423, 'RulingByPresentCourt': 31, 'None': 34}
[ensemble] predicted label counts: {'Fact': 494, 'Issue': 17, 'ArgumentPetitioner': 240, 'ArgumentRespondent': 14, 'Statute': 167, 'Dissent': 5, 'PrecedentReliedUpon': 410, 'PrecedentNotReliedUpon': 23, 'RulingByLowerCourt': 139, 'RatioOfTheDecision': 637, 'RulingByPresentCourt': 30, 'None': 21}
Saved: /kaggle/working/phase3_exp3_model2_lorma/prediction.json Prediction documents: 10


In [24]:
# ============================================================
# CELL 22 — Strict submission validation + ZIP
# Mirrors evaluator structural rules for private test data.
# ============================================================

def normalized_test_map(raw):
    out = {}
    for raw_id, sents in raw.items():
        did = normalize_id(raw_id)
        assert did not in out, f"Duplicate normalized ID: {did}"
        out[did] = sents
    return out

cl_test = normalized_test_map(cl_test_raw)
it_test = normalized_test_map(it_test_raw)

assert not (set(cl_test) & set(it_test)), "CL/IT test IDs overlap after normalization."

expected = {}
expected.update(cl_test)
expected.update(it_test)

assert set(prediction) == set(expected), (
    f"ID mismatch: missing={set(expected)-set(prediction)}, "
    f"extra={set(prediction)-set(expected)}"
)

allowed = set(LABELS)
for doc_id, sents in expected.items():
    pred = prediction[doc_id]
    assert isinstance(pred, list), doc_id
    assert len(pred) == len(sents), (
        doc_id, len(pred), len(sents)
    )
    assert all(x in allowed for x in pred), doc_id

with zipfile.ZipFile(
    SUBMISSION_ZIP,
    "w",
    zipfile.ZIP_DEFLATED
) as z:
    z.write(OFFICIAL_PRED, arcname="prediction.json")

with zipfile.ZipFile(SUBMISSION_ZIP, "r") as z:
    names = z.namelist()
    assert names == ["prediction.json"], names
    zipped = json.loads(z.read("prediction.json").decode("utf-8"))

assert zipped == prediction

print("✅ FINAL SUBMISSION PASSED STRUCTURAL CHECKS")
print("Documents:", len(prediction))
print("Sentences:", sum(len(v) for v in prediction.values()))
print("ZIP contents:", names)
print("prediction.json:", OFFICIAL_PRED)
print("submission.zip :", SUBMISSION_ZIP)

# ---- also write the two variants separately (upload both; keep the better)
def write_variant(name, pred):
    assert set(pred) == set(expected)
    for doc_id, sents in expected.items():
        assert len(pred[doc_id]) == len(sents), (name, doc_id)
        assert all(x in allowed for x in pred[doc_id]), (name, doc_id)
    jp = ROOT / f"prediction_{name}.json"
    zp = ROOT / f"submission_{name}.zip"
    with open(jp, "w", encoding="utf-8") as f:
        json.dump(pred, f, ensure_ascii=False)
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(jp, arcname="prediction.json")
    with zipfile.ZipFile(zp) as z:
        assert z.namelist() == ["prediction.json"]
    print("✅", zp.name)

for name, pred in predictions.items():
    write_variant(name, pred)
print("Variants written:", ", ".join(f"submission_{k}.zip" for k in predictions), "| submission.zip == refit")

# Sanity: how much do the variants disagree at sentence level?
def _flat(p): return [x for d in sorted(expected) for x in p[d]]
_names = list(predictions)
for i in range(len(_names)):
    for j in range(i + 1, len(_names)):
        a, b = _flat(predictions[_names[i]]), _flat(predictions[_names[j]])
        print(f"agreement {_names[i]} vs {_names[j]}: {np.mean([x == y for x, y in zip(a, b)]):.4f}")


# Evaluator-style structural contract check on TEST.
# Test gold labels are private, so only IDs, lengths and allowed labels can be verified.
def test_submission_contract(prediction_obj):
    if not isinstance(prediction_obj, dict):
        raise ValueError(
            "prediction.json must be a JSON object mapping document IDs to label lists."
        )

    normalized = {}

    for raw_id, labels in prediction_obj.items():
        did = normalize_id(raw_id)

        if did in normalized:
            raise ValueError(
                f"Duplicate document ID after .txt normalization: {did}"
            )

        normalized[did] = labels

    if set(normalized) != set(expected):
        missing = sorted(set(expected) - set(normalized))
        extra = sorted(set(normalized) - set(expected))
        raise ValueError(
            f"ID mismatch: missing={len(missing)}, extra={len(extra)}"
        )

    for did, source_sentences in expected.items():
        pred = normalized[did]

        if not isinstance(pred, list):
            raise ValueError(f"{did}: labels must be a JSON list.")

        if len(pred) != len(source_sentences):
            raise ValueError(
                f"{did}: sentence-count mismatch "
                f"({len(pred)} vs {len(source_sentences)})."
            )

        if any(x not in allowed for x in pred):
            raise ValueError(f"{did}: unknown label encountered.")

    return True

assert test_submission_contract(prediction)
print(
    "✅ TEST prediction satisfies the same ID / sentence-count / "
    "allowed-label contract enforced by the Phase-3 evaluator."
)


# Compact reproducibility manifest for future experiments/reporting.
manifest = {
    "experiment": "Phase3_Exp3_Model2_LORMA",
    "phase1_model": "Model2_DecoderOnly_Causal",
    "technique": "LORMA",
    "lorma_variant": "multiplicative_QV_only",
    "lorma_targets": ["Q", "V"],
    "lorma_r": LORMA_R,
    "lorma_alpha": LORMA_ALPHA,
    "lorma_dropout": LORMA_DROPOUT,
    "backbone_attention": "causal",
    "head_type": HEAD_TYPE,
    "head_hidden": HEAD_HIDDEN,
    "seed": SEED,
    "max_len": MAX_LEN,
    "context_sents_each_side": CONTEXT_SENTS,
    "micro_batch_size": MICRO_BATCH_SIZE,
    "grad_accum_steps": GRAD_ACCUM_STEPS,
    "effective_batch_size": MICRO_BATCH_SIZE * GRAD_ACCUM_STEPS,
    "lorma_lr": LORMA_LR,
    "head_lr": HEAD_LR,
    "weight_decay_head": WEIGHT_DECAY,
    "label_smoothing": LABEL_SMOOTHING,
    "class_weight_clip": [CLASS_WEIGHT_MIN, CLASS_WEIGHT_MAX],
    "best_epoch": int(best_obj["best_epoch"]),
    "best_epoch_frac": float(best_obj.get("best_frac", best_obj["best_epoch"])),
    "best_batch_in_epoch": int(best_obj.get("best_batch_in_epoch", 0)),
    "best_global_update": int(best_obj.get("best_global_update", 0)),
    "best_dev_macro_f1": float(best_obj["best_macro_f1"]),
    "decode_alpha": float(DECODE_ALPHA),
    "decode_beta": float(DECODE_BETA),
    "primary_submission": "submission.zip == refit",
    "tokenizer_sha256": sha256_file(TOKENIZER_PATH_RESOLVED),
    "phase1_checkpoint": str(PHASE1_CKPT),
    "artifacts": sorted([
        p.name for p in ROOT.iterdir()
        if p.is_file()
    ]),
}
with open(ROOT / "experiment_manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)
print("✅ Saved experiment manifest:", ROOT / "experiment_manifest.json")


✅ FINAL SUBMISSION PASSED STRUCTURAL CHECKS
Documents: 10
Sentences: 2197
ZIP contents: ['prediction.json']
prediction.json: /kaggle/working/phase3_exp3_model2_lorma/prediction.json
submission.zip : /kaggle/working/phase3_exp3_model2_lorma/submission.zip
✅ submission_heldout.zip
✅ submission_refit.zip
✅ submission_ensemble.zip
Variants written: submission_heldout.zip, submission_refit.zip, submission_ensemble.zip | submission.zip == refit
agreement heldout vs refit: 0.7010
agreement heldout vs ensemble: 0.8621
agreement refit vs ensemble: 0.8266
✅ TEST prediction satisfies the same ID / sentence-count / allowed-label contract enforced by the Phase-3 evaluator.


NameError: name 'TOKENIZER_PATH_RESOLVED' is not defined

## What to record for Experiment 3

After the run, keep the completed notebook and download:
- `best_lorma_rr.pt`
- `final_refit_lorma_rr.pt`
- `submission.zip` (primary refit copy)
- `submission_heldout.zip`
- `submission_refit.zip`
- `submission_ensemble.zip`
- `prediction_dev.json`
- `prediction_dev_decoded.json`
- `dev_logits.npz`, `test_logits.npz`
- `history.json`, `experiment_manifest.json`
- `evaluate_phase3.py`, `labels.json`, `cl_dev_truth.json`, `it_dev_truth.json`

For the report, record:

- Phase-1 architecture: **Model 2 decoder-only causal LM**
- Phase-3 technique: **LORMA / LoRMA+ multiplicative low-rank adaptation**
- Backbone LORMA targets: **Q and V only**
- `r=4`, `alpha=32`, adapter dropout `0.10`
- causal Phase-1 attention preserved
- downstream head: **BiGRU(256, bidirectional) + target/global pooling**
- exact trainable parameter count printed by Cell 10
- best held-out Macro-F1 and exact selected training position (epoch fraction / batch / update)
- selected `(alpha, beta)` transition-decoding setting
- elapsed time
- Codabench score for each submitted variant

### Run order

1. Attach `dec_full_best*.pt` and the exact tokenizer; T4 GPU on.
2. Run top-to-bottom.
3. Confirm strict Phase-1 checkpoint load succeeds.
4. Confirm the printed LORMA configuration is:
   `Q=True, K=False, V=True, r=4, alpha=32`, and backbone attention remains causal.
5. Confirm the LORMA parameter count and trainable names are sensible.
6. Let held-out training finish; do **not** manually toggle settings mid-run.
7. The exact supplied Phase-3 evaluator must pass before refit.
8. Run the remaining cells: dev decoding → fresh train+dev refit → test inference → structural checks → ZIPs.
9. If submission budget allows, submit **heldout**, **refit**, and **ensemble** separately and retain the best Codabench Macro-F1 as the Exp3 result.
